# 0. Analisis Eksploratif Poltrada V3 untuk Penelitian Knowledge Distillation YOLO26

Notebook mandiri ini mendokumentasikan sumber data, partisi, distribusi anotasi,
karakteristik citra, dan kualitas dataset **Poltrada V3** untuk skripsi
Fakultas Ilmu Komputer Universitas Brawijaya. Seluruh angka dan contoh citra
dihitung dari dataset yang ditambahkan melalui Kaggle Input ketika notebook dijalankan.
Notebook yang belum dieksekusi tidak memuat hasil empiris Poltrada V3.

Analisis mendukung deskripsi dataset pada Bab III dan dokumentasi serta interpretasi
hasil penelitian pada Bab V–VI. Karakteristik data dapat membantu menjelaskan konteks
evaluasi Knowledge Distillation (KD); karakteristik tersebut tidak membuktikan
keunggulan metode KD atau hubungan sebab-akibat.

**Ruang lingkup:** EDA deskriptif berbasis ground truth. Partisi test, bila tercantum
dan dapat diakses, hanya dideskripsikan. Hasilnya tidak digunakan untuk memilih
hyperparameter, checkpoint, atau metode KD. Tidak ada pelatihan, prediksi, pemuatan
model, penggunaan GPU, maupun integrasi W&B. Dataset dan notebook eksperimen 01–09
tidak ditulis ulang.

**Verifikasi repositori sebelum penyusunan:** konfigurasi notebook eksperimen 01–09
mengarah ke `/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml`
dan menetapkan `IMGSZ=640`. Loader eksperimen adalah `check_det_dataset` Ultralytics.
Isi YAML final dan direktori Kaggle belum tersedia di lingkungan pembuatan; pemeriksaan
aktual `nc`, `names`, partisi, dan pasangan image–label dilakukan pada Bagian 2–3.
Dataset lokal versi lain tidak digunakan sebagai pengganti Poltrada V3.

**Acuan penyajian:** bagian yang disebut dalam instruksi pengguna dari skripsi
Eldad Vikorian dan laporan final project Crowd Counting/KD. Dokumen asli kedua acuan
belum disertakan; pemetaan di bawah mengikuti deskripsi pengguna dan bukan klaim
verifikasi halaman. Tidak ada angka hasil kedua acuan yang disalin.

| Acuan menurut instruksi pengguna | Bagian notebook | Adaptasi untuk deteksi objek |
|---|---|---|
| Eldad, Bab III, Tabel 3.1–3.2 | 2, 4, 5 | Jumlah gambar dan instance per kelas serta partisi yang berasal dari YAML |
| Eldad, Bab V | 3, 7–12, 15 | Alur pembacaan data, audit, tabel mentah, dan dokumentasi anotasi |
| Eldad, Bab VI | 13, 14, 16 | Pemisahan train/validation/test dan batas interpretasi deskriptif |
| Laporan KD, 3.8.1 | 2–3, 13 | Inventaris YOLO, validasi anotasi, pasangan file, dan duplikasi |
| Laporan KD, 3.8.2–3.8.3 | 6–7 | Jumlah objek beranotasi, resolusi, dan kecerahan citra |
| Laporan KD, 3.8.4–3.8.5 | 8–10 | Pusat bounding box, luas relatif, jarak pusat, dan IoU |
| Laporan KD, 3.8.6 dan 3.8.8 | 12, 16 | Ground truth bounding box dan implikasi untuk penelitian KD |

Format input mengikuti [dokumentasi resmi dataset deteksi Ultralytics](https://docs.ultralytics.com/datasets/detect/).
Tidak digunakan density map Gaussian karena target penelitian berupa bounding box.

## 1. Konfigurasi dan lingkungan

### 1.1 Konfigurasi terpusat dan protokol eksekusi

Tambahkan **Poltrada V3 YOLO26 Clean** sebagai Kaggle Input. Pilih akselerator **None/CPU**,
kemudian jalankan **Run All**. Library ilmiah bawaan Kaggle digunakan tanpa instalasi
atau unduhan otomatis. Sel konfigurasi berikut merupakan satu-satunya tempat perubahan
parameter analisis. `standard` memindai seluruh label dan metadata, melakukan verifikasi
struktur file gambar, dan mengambil sampel brightness per split. `full` menambahkan
decoding lengkap seluruh gambar, brightness seluruh gambar, dan SHA-256 byte gambar.

`RUN_IMAGE_SHA256=None` mengikuti mode; nilai Boolean dapat mengubah pilihan tersebut.
Near-duplicate tetap opt-in karena kandidat pasangan bisa mahal. Sampling brightness
tidak digunakan untuk menghitung statistik anotasi. Ambang luas relatif bersifat
operasional, **bukan kategori COCO** dan bukan skala hasil resizing atau letterbox 640.

In [ ]:
from pathlib import Path

DATASET_NAME = "Poltrada V3"
DATA_YAML = Path(
    "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"
)
OUTPUT_DIR = Path("/kaggle/working/eda_poltrada_v3")
DATA_ROOT_OVERRIDE = (
    None  # Relokasi eksplisit jika YAML mencantumkan root yang tidak tersedia.
)
EDA_MODE = "standard"  # Pilihan: "standard" atau "full".
SEED = 42
BRIGHTNESS_SAMPLE_PER_SPLIT = 500
BRIGHTNESS_THUMBNAIL_SIDE = 256
RUN_IMAGE_SHA256 = None  # None: hanya aktif secara default pada mode full.
RUN_NEAR_DUPLICATES = False
NEAR_DUPLICATE_MAX_IMAGES = 1500  # Sampel total deterministik, bukan seluruh dataset.
DHASH_MAX_DISTANCE = 4
NEAR_DUPLICATE_MAX_PAIRS = 10_000  # Batas kandidat; truncation dilaporkan.
PAIRWISE_MAX_OBJECTS = (
    300  # Di atas batas ini seluruh gambar dilewati, bukan disubsampel.
)
RELATIVE_AREA_THRESHOLDS = (0.01, 0.05)
SPATIAL_BINS = 30
SAMPLE_GRID_SIZE = 8
FIGURE_DPI = 300
SAVE_SVG = True
COORDINATE_TOLERANCE = 1e-6  # Toleransi floating point untuk tepi kotak.


In [ ]:
import hashlib
import importlib.metadata as package_metadata
import json
import math
import platform
import textwrap
import time
from collections import Counter, defaultdict
from datetime import UTC, datetime
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
from PIL import Image, UnidentifiedImageError

if EDA_MODE not in {"standard", "full"}:
    raise ValueError("EDA_MODE wajib berupa 'standard' atau 'full'.")
if not (0 < RELATIVE_AREA_THRESHOLDS[0] < RELATIVE_AREA_THRESHOLDS[1] < 1):
    raise ValueError("Ambang luas relatif harus meningkat dan berada di (0, 1).")
if (
    min(
        BRIGHTNESS_SAMPLE_PER_SPLIT,
        BRIGHTNESS_THUMBNAIL_SIDE,
        PAIRWISE_MAX_OBJECTS,
        SPATIAL_BINS,
        SAMPLE_GRID_SIZE,
        NEAR_DUPLICATE_MAX_IMAGES,
        NEAR_DUPLICATE_MAX_PAIRS,
    )
    < 1
):
    raise ValueError("Ukuran sampel, batas pasangan, dan jumlah bin harus positif.")
if not 0 <= DHASH_MAX_DISTANCE <= 64 or FIGURE_DPI < 200:
    raise ValueError("Jarak dHash harus 0–64; resolusi ekspor minimal 200 DPI.")
if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"YAML dataset tidak ditemukan: {DATA_YAML}. Tambahkan dataset final melalui "
        "Kaggle Input; notebook tidak mengunduh atau mengganti sumber secara otomatis."
    )
DATA_YAML = DATA_YAML.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()
input_root = Path("/kaggle/input").resolve()
if OUTPUT_DIR.is_relative_to(input_root) or DATA_YAML.is_relative_to(OUTPUT_DIR):
    raise ValueError(
        "Output tidak boleh berada di Kaggle Input atau menaungi sumber YAML."
    )
RUN_STARTED = datetime.now(UTC).isoformat()
START_TIME = time.perf_counter()
HASH_ENABLED = (
    EDA_MODE == "full" if RUN_IMAGE_SHA256 is None else bool(RUN_IMAGE_SHA256)
)
VERSIONS = {
    name: package_metadata.version(name)
    for name in ["numpy", "pandas", "matplotlib", "Pillow", "PyYAML", "ipython"]
}
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "figure.dpi": 110,
        "savefig.dpi": FIGURE_DPI,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
ISSUE_ROWS: list[dict[str, Any]] = []
TABLES: dict[str, pd.DataFrame] = {}
FIGURE_RECORDS: list[dict[str, Any]] = []
FINDINGS: list[dict[str, str]] = []
TIMINGS: dict[str, float] = {}
display(
    pd.DataFrame(
        [
            {
                "python": platform.python_version(),
                "mode": EDA_MODE,
                "yaml": str(DATA_YAML),
                "output": str(OUTPUT_DIR),
                "seed": SEED,
                "sha256_gambar": HASH_ENABLED,
                "near_duplicate": RUN_NEAR_DUPLICATES,
            }
        ]
    )
)
display(pd.DataFrame(VERSIONS.items(), columns=["library", "versi"]))


### 1.2 Fungsi bersama untuk audit, statistik, dan ekspor

Statistik numerik menggunakan simpangan baku sampel (`ddof=1`), kuartil interpolasi
linear Pandas, serta IQR = Q3 − Q1. Nilai tidak tersedia (`NaN`) tidak diubah menjadi
nol. Interpretasi dicatat untuk laporan akhir. Tabel mentah tetap diekspor agar angka
dapat dihitung ulang. Grafik disimpan dengan 300 DPI dan SVG bila diaktifkan.

In [ ]:
def add_issue(
    split: str,
    file: str | Path,
    issue: str,
    severity: str,
    detail: str,
    line: int | None = None,
    occurrences: int = 1,
) -> None:
    """Tambahkan temuan tanpa mengubah sumber; occurrences adalah banyak kejadian."""
    ISSUE_ROWS.append(
        {
            "split": split,
            "file": str(file),
            "issue": issue,
            "severity": severity,
            "line": line,
            "occurrences": occurrences,
            "detail": detail,
        }
    )


def register_table(name: str, frame: pd.DataFrame, show: bool = True) -> pd.DataFrame:
    """Simpan tabel untuk ekspor tertunda dan tampilkan maksimal 30 baris."""
    TABLES[name] = frame.copy()
    if show:
        display(frame.head(30))
        if len(frame) > 30:
            display(
                Markdown(f"Tabel memuat {len(frame):,} baris; seluruh baris diekspor.")
            )
    return frame


def describe_numeric(
    frame: pd.DataFrame, columns: list[str], group: str | None = None
) -> pd.DataFrame:
    """Hitung count, mean, median, SD, rentang, kuartil, IQR, P5, dan P95."""
    rows = []
    groups = [("seluruh", frame)] if group is None else frame.groupby(group, sort=False)
    for group_name, part in groups:
        for column in columns:
            values = (
                pd.to_numeric(part[column], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )
            q1, q3 = values.quantile([0.25, 0.75])
            rows.append(
                {
                    "kelompok": group_name,
                    "variabel": column,
                    "count": len(values),
                    "mean": values.mean(),
                    "median": values.median(),
                    "std": values.std(ddof=1),
                    "min": values.min(),
                    "max": values.max(),
                    "q1": q1,
                    "q3": q3,
                    "iqr": q3 - q1,
                    "p5": values.quantile(0.05),
                    "p95": values.quantile(0.95),
                }
            )
    return pd.DataFrame(
        rows,
        columns=[
            "kelompok",
            "variabel",
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
            "q1",
            "q3",
            "iqr",
            "p5",
            "p95",
        ],
    )


def interpret(code_id: str, text: str) -> None:
    """Tampilkan narasi dengan angka runtime dan simpan dalam laporan."""
    FINDINGS.append({"kode": code_id, "interpretasi": text})
    display(Markdown(f"**Interpretasi ({code_id}):** {text}"))


def finish_figure(fig: plt.Figure, name: str, caption: str) -> None:
    """Simpan PNG/SVG, tampilkan, dan selalu tutup figure Matplotlib."""
    directory = OUTPUT_DIR / "figures"
    directory.mkdir(parents=True, exist_ok=True)
    try:
        fig.tight_layout()
        extensions = ["png", "svg"] if SAVE_SVG else ["png"]
        for extension in extensions:
            target = directory / f"{name}.{extension}"
            fig.savefig(target, dpi=FIGURE_DPI, bbox_inches="tight")
        FIGURE_RECORDS.append(
            {"name": name, "caption": caption, "formats": extensions, "dpi": FIGURE_DPI}
        )
        display(fig)
        display(Markdown(f"**Gambar {name}:** {caption}"))
    finally:
        plt.close(fig)


def empty_axis(
    ax: plt.Axes, text: str = "Data yang memenuhi syarat tidak tersedia"
) -> None:
    """Tampilkan alasan ketika distribusi kosong, tanpa membuat data pengganti."""
    ax.text(0.5, 0.5, text, transform=ax.transAxes, ha="center", va="center", wrap=True)


def sha256_file(path: Path) -> str:
    """Hitung SHA-256 secara streaming; tidak memuat file utuh ke RAM."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


## 2. Sumber dan struktur dataset

### 2.1 Pembacaan YAML, kelas, dan resolusi partisi

**Tujuan:** Memastikan identitas kelas dan partisi berasal dari YAML final.

**Metode:** YAML dibaca dengan safe loader yang menolak key duplikat. `nc` wajib
integer positif dan sama dengan jumlah nama kelas. ID dictionary `names` wajib
kontigu 0…nc−1. Nama kelas kosong atau ganda ditolak agar agregasi tidak ambigu.
Root relatif diselesaikan terhadap direktori YAML; root absolut dipertahankan.
Relokasi hanya dilakukan dengan `DATA_ROOT_OVERRIDE` dan dicatat. `train`, `val`,
dan `test` mendukung direktori, daftar sumber, atau file daftar gambar. Untuk split
string `../...` yang tidak ada, fallback menghilangkan satu awalan `../`, sesuai
loader eksperimen Ultralytics, dan dicatat secara eksplisit.

Dalam daftar gambar, awalan `./` relatif ke file daftar; path relatif lain mengikuti
direktori kerja seperti loader eksperimen. Tidak ada pencarian otomatis ke dataset
lain. Partisi yang dideklarasikan tetapi hilang dilaporkan sebagai error, bukan
dianggap partisi opsional yang tidak tersedia. Mapping label mengganti segmen
`images` terakhir menjadi `labels`, lalu mengganti ekstensi dengan `.txt`.

**Caption:** Tabel sumber menampilkan nilai YAML, sumber teresolusi, dan cakupan audit
label. Seluruh label dalam direktori pasangan diperiksa. Untuk daftar subset, label
milik gambar di luar subset dilaporkan sebagai *di luar keanggotaan split*, bukan
label yatim bila gambar pasangannya ada pada direktori sumber.

**Interpretasi dan implikasi:** Daftar kelas runtime menentukan urutan seluruh
tabel, legenda, dan warna. Nama yang ternyata bukan kategori kendaraan tetap
dipertahankan; istilah objek beranotasi digunakan tanpa menebak ontologi kelas.

In [ ]:
class UniqueKeyLoader(yaml.SafeLoader):
    """Safe YAML loader yang menolak key duplikat, termasuk dictionary names."""


def construct_unique_mapping(
    loader: UniqueKeyLoader, node: Any, deep: bool = False
) -> dict:
    mapping = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise ValueError(f"Key YAML duplikat: {key!r}")
        mapping[key] = loader.construct_object(value_node, deep=deep)
    return mapping


UniqueKeyLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, construct_unique_mapping
)
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
    ".jp2",
    ".avif",
    ".heic",
    ".dng",
    ".mpo",
}
SPLIT_LABELS = {"train": "Pelatihan", "val": "Validasi", "test": "Pengujian"}
SPLIT_COLORS = {"train": "#0072B2", "val": "#E69F00", "test": "#009E73"}


def validate_class_schema(data: dict) -> list[str]:
    """Kembalikan names berurutan hanya bila nc, ID, dan nama konsisten."""
    nc = data.get("nc")
    if isinstance(nc, bool) or not isinstance(nc, int) or nc < 1:
        raise ValueError("YAML wajib menyediakan nc sebagai integer positif.")
    names = data.get("names")
    if isinstance(names, dict):
        indexed = {}
        for key, value in names.items():
            if isinstance(key, bool) or str(key) != str(int(key)):
                raise ValueError(f"ID names tidak berupa integer kanonik: {key!r}")
            if int(key) in indexed:
                raise ValueError("ID names ganda setelah normalisasi integer.")
            indexed[int(key)] = value
        if set(indexed) != set(range(nc)):
            raise ValueError("ID names wajib kontigu mulai 0 dan sesuai nc.")
        names = [indexed[index] for index in range(nc)]
    if not isinstance(names, list) or len(names) != nc:
        raise ValueError("Jumlah names harus sesuai nc; names berupa list/dictionary.")
    if any(not isinstance(name, str) or not name.strip() for name in names):
        raise ValueError("Setiap nama kelas wajib string tidak kosong.")
    if len(set(names)) != nc:
        raise ValueError("Nama kelas ganda; agregasi berdasarkan kelas menjadi ambigu.")
    return names


def image_to_label(image_path: Path) -> Path:
    """Petakan segmen images terakhir ke labels; fallback label berdampingan."""
    parts = list(image_path.parts)
    indices = [index for index, part in enumerate(parts) if part == "images"]
    if indices:
        parts[indices[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def discover_split(
    value: Any, root: Path, split: str
) -> tuple[list[Path], list[Path], list[dict]]:
    """Inventaris direktori/daftar tanpa mengubah sumber; catat referensi berulang."""
    references, label_roots, source_rows = [], set(), []
    entries = value if isinstance(value, list) else [value]
    if not entries:
        raise ValueError(f"Split {split} dideklarasikan sebagai daftar kosong.")
    for entry in entries:
        if not isinstance(entry, str) or not entry.strip():
            raise ValueError(
                f"Entri split {split} harus berupa path string tidak kosong."
            )
        source = (root / entry).resolve()
        resolution = "path YAML"
        if not source.exists() and isinstance(value, str) and entry.startswith("../"):
            fallback = (root / entry[3:]).resolve()
            if fallback.exists():
                source = fallback
                resolution = "fallback ../ Ultralytics; sumber tidak diubah"
                add_issue(split, source, "fallback_path_yaml", "INFO", resolution)
        if not source.exists():
            raise FileNotFoundError(
                f"Split {split} dideklarasikan tetapi sumber hilang: {source}"
            )
        if source.is_dir():
            files = sorted(
                path.resolve()
                for path in source.rglob("*")
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            )
            label_roots.add(image_to_label(source / "__probe__.jpg").parent)
            source_type = "direktori"
        elif source.suffix.lower() == ".txt":
            files = []
            for line_number, line in enumerate(
                source.read_text(encoding="utf-8-sig").splitlines(), 1
            ):
                line = line.strip()
                if not line:
                    continue
                path = (
                    (source.parent / line[2:]) if line.startswith("./") else Path(line)
                )
                path = path.resolve()
                if path.suffix.lower() not in IMAGE_EXTENSIONS:
                    add_issue(
                        split,
                        source,
                        "ekstensi_daftar_tidak_didukung",
                        "WARNING",
                        line,
                        line_number,
                    )
                    continue
                files.append(path)
                label_roots.add(image_to_label(path).parent)
            source_type = "daftar gambar"
        else:
            raise ValueError(f"Sumber split bukan direktori atau daftar .txt: {source}")
        references.extend(files)
        source_rows.append(
            {
                "split": split,
                "nilai_yaml": entry,
                "sumber": str(source),
                "jenis": source_type,
                "resolusi": resolution,
                "referensi_gambar": len(files),
            }
        )
    counts = Counter(references)
    for path, count in counts.items():
        if count > 1:
            add_issue(
                split,
                path,
                "referensi_gambar_berulang",
                "WARNING",
                "Keanggotaan unik digunakan untuk EDA; daftar asli tidak diubah.",
                occurrences=count - 1,
            )
    return sorted(counts), sorted(label_roots), source_rows


def load_dataset(yaml_path: Path, root_override: str | Path | None = None) -> tuple:
    """Baca schema dan split; test yang tidak dideklarasikan tidak dibuat."""
    with yaml_path.open(encoding="utf-8-sig") as handle:
        data = yaml.load(handle, Loader=UniqueKeyLoader)
    if not isinstance(data, dict):
        raise TypeError("Isi YAML dataset harus mapping.")
    names = validate_class_schema(data)
    raw_root = data.get("path") or "."
    root = (
        Path(root_override).resolve()
        if root_override is not None
        else (yaml_path.parent / str(raw_root)).resolve()
    )
    if not root.is_dir():
        raise FileNotFoundError(
            f"Root YAML tidak tersedia: {root}. Periksa DATA_ROOT_OVERRIDE."
        )
    splits, roots, sources = {}, {}, []
    for split in ("train", "val", "test"):
        value = data.get(split)
        if value is None or value == "":
            if split != "test":
                raise ValueError(f"YAML tidak menyediakan split wajib {split}.")
            continue
        paths, label_roots, rows = discover_split(value, root, split)
        if not paths:
            add_issue(
                split,
                root,
                "split_kosong",
                "CRITICAL",
                "Tidak ada gambar dalam sumber.",
            )
        splits[split], roots[split] = paths, label_roots
        sources.extend(rows)
    return data, root, names, splits, roots, pd.DataFrame(sources)


In [ ]:
try:
    (
        DATA_CONFIG,
        DATA_ROOT,
        CLASS_NAMES,
        SPLIT_IMAGES,
        SPLIT_LABEL_ROOTS,
        SOURCE_TABLE,
    ) = load_dataset(DATA_YAML, DATA_ROOT_OVERRIDE)
except (OSError, ValueError, TypeError, yaml.YAMLError) as error:
    # Simpan kegagalan schema sebelum menghentikan analisis yang tidak aman.
    report_dir = OUTPUT_DIR / "reports"
    report_dir.mkdir(parents=True, exist_ok=True)
    schema_report = pd.DataFrame(
        [
            {
                "file": str(DATA_YAML),
                "issue": "schema_atau_path_yaml",
                "severity": "CRITICAL",
                "occurrences": 1,
                "detail": str(error),
            }
        ]
    )
    schema_report.to_csv(
        report_dir / "audit_report.csv", index=False, encoding="utf-8-sig"
    )
    raise ValueError(
        f"Validasi dataset gagal; laporan disimpan di {report_dir}: {error}"
    ) from error

if OUTPUT_DIR.is_relative_to(DATA_ROOT) or DATA_ROOT.is_relative_to(OUTPUT_DIR):
    raise ValueError("Direktori output harus terpisah dari root dataset.")
for source in SOURCE_TABLE["sumber"]:
    source_path = Path(source)
    if OUTPUT_DIR.is_relative_to(source_path) or source_path.is_relative_to(OUTPUT_DIR):
        raise ValueError("Output tidak boleh beririsan dengan sumber split.")
SPLITS = list(SPLIT_IMAGES)
CLASS_IDS = list(range(len(CLASS_NAMES)))
palette = plt.get_cmap("tab10" if len(CLASS_NAMES) <= 10 else "tab20")
CLASS_COLORS = {index: palette(index % palette.N) for index in CLASS_IDS}
register_table("dataset_sources", SOURCE_TABLE)
register_table(
    "class_schema", pd.DataFrame({"class_id": CLASS_IDS, "nama_kelas": CLASS_NAMES})
)
display(
    Markdown(
        f"**Root teresolusi:** `{DATA_ROOT}`. **nc:** {len(CLASS_NAMES)}. "
        f"**Partisi:** {', '.join(SPLITS)}. "
        + (
            "Test tersedia dan dianalisis secara deskriptif."
            if "test" in SPLITS
            else "**Test tidak tercantum dalam YAML; tidak dibuat split baru.**"
        )
    )
)
display(Markdown("**YAML asli (dibaca tanpa penulisan ulang):**"))
print(DATA_YAML.read_text(encoding="utf-8-sig"))


## 3. Inventarisasi dan integritas data

### 3.1 Parser dan aturan validitas anotasi YOLO

**Tujuan:** Menilai kualitas anotasi tanpa memperbaiki atau menghapus sumber.

**Metode:** Setiap baris nonkosong harus mempunyai lima kolom
`class_id x_center y_center width height`. Parser memeriksa tipe numerik, NaN/Inf,
ID integer/rentang, koordinat [0,1], dimensi positif, dan tepi kotak pada [0,1]
dengan toleransi numerik yang dinyatakan. Kotak identik dilaporkan berdasarkan
kesamaan tuple numerik lima kolom. Kotak identik yang valid tetap dihitung sehingga
angka merepresentasikan isi file, bukan hasil deduplikasi loader training.

**Caption:** Tabel audit mencatat file, split, baris, jenis masalah, severity, dan
jumlah kejadian. Baris mentah serta alasan kelayakan analisis disimpan terpisah.

**Interpretasi:** File label kosong merupakan kandidat background valid jika
gambarnya dapat diverifikasi. File label hilang diperlakukan sebagai status tidak
diketahui: konvensi YOLO mengizinkan background tanpa label, tetapi ketiadaan file
sendiri tidak membuktikan tidak adanya objek. File rusak dan background dibedakan.

**Implikasi:** Distribusi kelas memakai baris geometris valid dari pasangan unik.
Distribusi jumlah objek dan komposisi kelas memakai gambar dengan label lengkap
tanpa baris tidak valid. Denominator tersebut dilaporkan; temuan kritis perlu
ditindaklanjuti tanpa otomatis membatalkan semua eksperimen.

In [ ]:
def parse_yolo_text(
    text: str, nc: int, tolerance: float = 1e-6
) -> tuple[list[dict], list[dict]]:
    """Audit teks YOLO; simpan setiap baris, tanpa clipping atau deduplikasi."""
    records, issues, seen = [], [], set()
    for line_number, raw_line in enumerate(text.splitlines(), 1):
        raw_line = raw_line.strip()
        if not raw_line:
            continue
        tokens = raw_line.split()
        problems = []
        record = {
            "line": line_number,
            "raw_text": raw_line,
            "class_id": None,
            "x_center": np.nan,
            "y_center": np.nan,
            "width_norm": np.nan,
            "height_norm": np.nan,
            "valid": False,
            "duplicate": False,
        }
        if len(tokens) != 5:
            problems.append("jumlah_kolom_salah")
        else:
            try:
                values = [float(token) for token in tokens]
            except ValueError:
                problems.append("nilai_bukan_numerik")
            else:
                class_value, x_center, y_center, width, height = values
                record.update(
                    x_center=x_center,
                    y_center=y_center,
                    width_norm=width,
                    height_norm=height,
                )
                if not all(math.isfinite(value) for value in values):
                    problems.append("nilai_nan_inf")
                else:
                    if not class_value.is_integer():
                        problems.append("class_id_bukan_integer")
                    elif not 0 <= int(class_value) < nc:
                        problems.append("class_id_di_luar_rentang")
                    else:
                        record["class_id"] = int(class_value)
                    if not all(0 <= value <= 1 for value in values[1:]):
                        problems.append("koordinat_normalisasi_tidak_valid")
                    if width <= 0 or height <= 0:
                        problems.append("dimensi_nol_atau_negatif")
                    if (
                        x_center - width / 2 < -tolerance
                        or y_center - height / 2 < -tolerance
                        or x_center + width / 2 > 1 + tolerance
                        or y_center + height / 2 > 1 + tolerance
                    ):
                        problems.append("bbox_melewati_batas")
                    key = tuple(values)
                    if key in seen:
                        record["duplicate"] = True
                        issues.append(
                            {
                                "issue": "bbox_identik_terduplikasi",
                                "severity": "WARNING",
                                "line": line_number,
                                "detail": raw_line,
                            }
                        )
                    seen.add(key)
        record["valid"] = not problems
        record["invalid_reasons"] = "|".join(problems)
        records.append(record)
        issues.extend(
            {
                "issue": issue,
                "severity": "CRITICAL",
                "line": line_number,
                "detail": raw_line,
            }
            for issue in problems
        )
    return records, issues


def xywh_to_pixels(
    x_center: float,
    y_center: float,
    width: float,
    height: float,
    image_width: int,
    image_height: int,
) -> tuple[float, float, float, float]:
    """Konversi normalized xywh ke xyxy piksel asli tanpa clipping/rotasi."""
    return (
        (x_center - width / 2) * image_width,
        (y_center - height / 2) * image_height,
        (x_center + width / 2) * image_width,
        (y_center + height / 2) * image_height,
    )


def bbox_geometry(
    width_norm: float, height_norm: float, image_width: float, image_height: float
) -> dict[str, float]:
    """Luas relatif=w*h; aspect ratio fisik=(w*W)/(h*H), bukan w/h."""
    width_px, height_px = width_norm * image_width, height_norm * image_height
    return {
        "area_relative": width_norm * height_norm,
        "width_px": width_px,
        "height_px": height_px,
        "area_px": width_px * height_px,
        "aspect_ratio_pixel": width_px / height_px if height_px > 0 else np.nan,
        "aspect_ratio_normalized": width_norm / height_norm,
    }


### 3.2 Pemindaian seluruh file dan cakupan integritas

Semua gambar diinventarisasi satu per satu; pixel tidak disimpan secara kolektif.
Mode standard menggunakan Pillow `verify()` untuk integritas struktur file.
Mode full juga menjalankan `load()` untuk deteksi kegagalan decoding. Keberhasilan
`verify()` saja tidak membuktikan seluruh pixel JPEG dapat didekode. Orientasi EXIF
nonstandar ditandai; statistik dan visualisasi memakai koordinat pixel mentah,
tanpa rotasi otomatis. Pasangan ambigu (dua gambar memakai label yang sama) dikeluarkan
dari agregasi anotasi dan tetap dilaporkan. Label yatim dibaca untuk audit namun
tidak dihitung sebagai instance dataset.

In [ ]:
RAW_COLUMNS = [
    "image_id",
    "split",
    "image_path",
    "label_path",
    "line",
    "raw_text",
    "class_id",
    "x_center",
    "y_center",
    "width_norm",
    "height_norm",
    "valid",
    "duplicate",
    "invalid_reasons",
    "analysis_eligible",
]
IMAGE_COLUMNS = [
    "image_id",
    "split",
    "image_path",
    "label_path",
    "file_bytes",
    "width_px",
    "height_px",
    "image_aspect_ratio",
    "readable",
    "integrity_level",
    "exif_orientation",
    "label_status",
    "raw_rows",
    "invalid_rows",
    "n_valid_objects",
    "n_objects",
    "annotation_complete",
    "pair_ambiguous",
    "brightness",
    "brightness_sampled",
    "sha256",
    "dhash",
]
LABEL_COLUMNS = [
    "split",
    "label_path",
    "paired_images",
    "status",
    "raw_rows",
    "invalid_rows",
    "sha256",
    "scope",
]
ISSUE_COLUMNS = ["split", "file", "issue", "severity", "line", "occurrences", "detail"]


def inspect_image(path: Path, full_decode: bool) -> dict:
    """Periksa struktur/pixel secara opsional; kembalikan error yang dapat diaudit."""
    result = {
        "file_bytes": np.nan,
        "width_px": np.nan,
        "height_px": np.nan,
        "image_aspect_ratio": np.nan,
        "readable": False,
        "integrity_level": "belum diperiksa",
        "exif_orientation": None,
        "error": None,
    }
    try:
        result["file_bytes"] = path.stat().st_size
        with Image.open(path) as image:
            width, height = image.size
            orientation = image.getexif().get(274, 1)
        # getexif() dapat mendekode PNG; verify() wajib memakai objek yang baru dibuka.
        with Image.open(path) as image:
            image.verify()
        if full_decode:
            with Image.open(path) as image:
                image.load()
        result.update(
            width_px=width,
            height_px=height,
            image_aspect_ratio=width / height,
            readable=True,
            exif_orientation=orientation,
            integrity_level="decode lengkap" if full_decode else "verifikasi struktur",
        )
    except (
        OSError,
        ValueError,
        SyntaxError,
        UnidentifiedImageError,
        Image.DecompressionBombError,
    ) as error:
        result["error"] = f"{type(error).__name__}: {error}"
    return result


def label_has_image_partner(label_path: Path) -> bool:
    """Cari pasangan fisik label di direktori images dengan stem/ekstensi didukung."""
    parts = list(label_path.parts)
    indices = [index for index, part in enumerate(parts) if part == "labels"]
    if indices:
        parts[indices[-1]] = "images"
    parent = Path(*parts).parent
    if not parent.is_dir():
        return False
    return any(
        path.stem == label_path.stem and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in parent.iterdir()
        if path.is_file()
    )


def scan_dataset() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Audit metadata dan seluruh label; validitas baris terpisah dari integritas citra."""
    images, raw_records, labels = [], [], []
    for split, paths in SPLIT_IMAGES.items():
        pairing = defaultdict(list)
        for image_path in paths:
            pairing[image_to_label(image_path)].append(image_path)
        candidates = set(pairing)
        for label_root in SPLIT_LABEL_ROOTS[split]:
            if label_root.is_dir():
                candidates.update(path.resolve() for path in label_root.rglob("*.txt"))
        parsed = {}
        for label_path in sorted(candidates):
            paired_count = len(pairing.get(label_path, []))
            if paired_count > 1:
                add_issue(
                    split,
                    label_path,
                    "pasangan_label_ambigu",
                    "CRITICAL",
                    "Lebih dari satu gambar memakai label yang sama.",
                    occurrences=paired_count,
                )
            scope = (
                "anggota split"
                if paired_count
                else (
                    "di luar keanggotaan split"
                    if label_has_image_partner(label_path)
                    else "label yatim"
                )
            )
            if scope == "label yatim":
                add_issue(
                    split,
                    label_path,
                    "label_tanpa_gambar",
                    "WARNING",
                    "Tidak ada pasangan fisik.",
                )
            elif scope == "di luar keanggotaan split":
                add_issue(
                    split,
                    label_path,
                    "label_di_luar_keanggotaan",
                    "INFO",
                    "Gambar pasangan ada, tetapi tidak dipilih sumber split ini.",
                )
            digest = None
            if not label_path.is_file():
                status, records = "label hilang", []
                add_issue(
                    split,
                    label_path,
                    "gambar_tanpa_label",
                    "WARNING",
                    "Background tanpa file mungkin valid menurut YOLO; perlu verifikasi maksud anotasi.",
                )
            else:
                try:
                    payload = label_path.read_bytes()
                    digest = hashlib.sha256(payload).hexdigest()
                    records, issues = parse_yolo_text(
                        payload.decode("utf-8-sig"),
                        len(CLASS_NAMES),
                        COORDINATE_TOLERANCE,
                    )
                    for issue in issues:
                        add_issue(split, label_path, **issue)
                    status = (
                        "label kosong"
                        if not records
                        else "label tidak valid"
                        if any(not item["valid"] for item in records)
                        else "label valid"
                    )
                except (OSError, UnicodeError) as error:
                    status, records = "label tidak terbaca", []
                    add_issue(
                        split, label_path, "label_tidak_terbaca", "CRITICAL", str(error)
                    )
            parsed[label_path] = (status, records)
            labels.append(
                {
                    "split": split,
                    "label_path": str(label_path),
                    "paired_images": paired_count,
                    "status": status,
                    "raw_rows": len(records),
                    "invalid_rows": sum(not item["valid"] for item in records),
                    "sha256": digest,
                    "scope": scope,
                }
            )
            if not paired_count:
                raw_records.extend(
                    {
                        **item,
                        "image_id": None,
                        "split": split,
                        "image_path": None,
                        "label_path": str(label_path),
                        "analysis_eligible": False,
                    }
                    for item in records
                )
        for index, image_path in enumerate(paths):
            image_id = f"{split}:{index:08d}"
            label_path = image_to_label(image_path)
            status, records = parsed[label_path]
            ambiguous = len(pairing[label_path]) > 1
            complete = status in {"label valid", "label kosong"} and not ambiguous
            metadata = inspect_image(image_path, EDA_MODE == "full")
            error = metadata.pop("error")
            if error:
                add_issue(split, image_path, "gambar_tidak_terbaca", "CRITICAL", error)
            if metadata["exif_orientation"] not in (None, 1):
                add_issue(
                    split,
                    image_path,
                    "orientasi_exif_nonstandar",
                    "WARNING",
                    "Analisis memakai pixel mentah; periksa konvensi orientasi anotasi.",
                )
            valid_count = sum(item["valid"] for item in records)
            images.append(
                {
                    **metadata,
                    "image_id": image_id,
                    "split": split,
                    "image_path": str(image_path),
                    "label_path": str(label_path),
                    "label_status": status,
                    "raw_rows": len(records),
                    "invalid_rows": sum(not item["valid"] for item in records),
                    "n_valid_objects": valid_count if not ambiguous else np.nan,
                    "n_objects": valid_count if complete else np.nan,
                    "annotation_complete": complete,
                    "pair_ambiguous": ambiguous,
                    "brightness": np.nan,
                    "brightness_sampled": False,
                    "sha256": None,
                    "dhash": None,
                }
            )
            raw_records.extend(
                {
                    **item,
                    "image_id": image_id,
                    "split": split,
                    "image_path": str(image_path),
                    "label_path": str(label_path),
                    "analysis_eligible": bool(item["valid"] and not ambiguous),
                }
                for item in records
            )
        print(
            f"{SPLIT_LABELS[split]}: {len(paths):,} gambar dan {len(candidates):,} path label diperiksa."
        )
    return (
        pd.DataFrame(images, columns=IMAGE_COLUMNS),
        pd.DataFrame(raw_records, columns=RAW_COLUMNS),
        pd.DataFrame(labels, columns=LABEL_COLUMNS),
    )


In [ ]:
scan_started = time.perf_counter()
images_df, raw_labels_df, labels_df = scan_dataset()
TIMINGS["inventaris_dan_label"] = time.perf_counter() - scan_started
if images_df.empty:
    register_table("annotation_issues", pd.DataFrame(ISSUE_ROWS, columns=ISSUE_COLUMNS))
    (OUTPUT_DIR / "reports").mkdir(parents=True, exist_ok=True)
    TABLES["annotation_issues"].to_csv(
        OUTPUT_DIR / "reports/audit_report.csv", index=False
    )
    raise ValueError(
        "Tidak ada gambar terinventarisasi; audit disimpan dan analisis dihentikan."
    )
boxes_df = raw_labels_df.loc[raw_labels_df["analysis_eligible"]].copy()
boxes_df["class_id"] = boxes_df["class_id"].astype(int)
boxes_df["class_name"] = boxes_df["class_id"].map(dict(enumerate(CLASS_NAMES)))
boxes_df = boxes_df.merge(
    images_df[["image_id", "width_px", "height_px"]],
    on="image_id",
    how="left",
    validate="many_to_one",
)
boxes_df["area_relative"] = boxes_df["width_norm"] * boxes_df["height_norm"]
boxes_df["width_box_px"] = boxes_df["width_norm"] * boxes_df["width_px"]
boxes_df["height_box_px"] = boxes_df["height_norm"] * boxes_df["height_px"]
boxes_df["area_px"] = boxes_df["width_box_px"] * boxes_df["height_box_px"]
boxes_df["aspect_ratio_pixel"] = boxes_df["width_box_px"] / boxes_df["height_box_px"]
boxes_df["aspect_ratio_normalized"] = boxes_df["width_norm"] / boxes_df["height_norm"]
small_threshold, medium_threshold = RELATIVE_AREA_THRESHOLDS
boxes_df["size_relative"] = pd.cut(
    boxes_df["area_relative"],
    bins=[0, small_threshold, medium_threshold, np.inf],
    labels=["Kecil relatif", "Sedang relatif", "Besar relatif"],
    right=False,
)
count_images_df = images_df.loc[images_df["annotation_complete"]].copy()
initial_audit = pd.DataFrame(ISSUE_ROWS, columns=ISSUE_COLUMNS)
register_table("initial_annotation_audit", initial_audit, show=False)
display(
    initial_audit.groupby(["severity", "issue"], sort=False)["occurrences"]
    .sum()
    .reset_index()
)
critical_count = int(
    initial_audit.loc[initial_audit["severity"].eq("CRITICAL"), "occurrences"].sum()
)
interpret(
    "AUDIT-AWAL",
    f"Terinventarisasi {len(images_df):,} gambar dan {len(labels_df):,} "
    f"path label; {len(boxes_df):,} baris bbox valid dari pasangan unik. "
    f"Ditemukan {critical_count:,} kejadian kritis. Durasi scan "
    f"{TIMINGS['inventaris_dan_label']:.2f} detik. "
    "Temuan tidak mengubah sumber dan tidak otomatis membatalkan eksperimen.",
)


## 4. Pembagian dataset

### 4.1 Jumlah gambar dan instance per partisi — EDA-01

**Tujuan:** Mendokumentasikan pembagian dataset penelitian yang benar-benar tersedia.

**Metode:** Jumlah gambar berasal dari keanggotaan path unik setiap split. Instance adalah baris bbox valid dari pasangan unik; background adalah label kosong pada gambar terverifikasi. Proporsi split menggunakan seluruh gambar terinventarisasi sebagai denominator.

**Caption:** Jumlah gambar per split serta tabel instance valid, label tidak diketahui/tidak valid, dan integritas citra.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Tabel mendukung Bab III; partisi test tetap terpisah dari seleksi model.

In [ ]:
split_rows = []
for split in SPLITS:
    part = images_df.loc[images_df["split"].eq(split)]
    split_rows.append(
        {
            "split": split,
            "gambar": len(part),
            "proporsi_gambar_pct": 100 * len(part) / len(images_df),
            "instance_valid": int(boxes_df["split"].eq(split).sum()),
            "gambar_label_lengkap": int(part["annotation_complete"].sum()),
            "background_label_kosong_terverifikasi": int(
                (
                    part["label_status"].eq("label kosong")
                    & part["readable"]
                    & ~part["pair_ambiguous"]
                ).sum()
            ),
            "label_hilang": int(part["label_status"].eq("label hilang").sum()),
            "gambar_tidak_terbaca": int((~part["readable"]).sum()),
        }
    )
split_summary = register_table("split_summary", pd.DataFrame(split_rows))
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [SPLIT_LABELS[split] for split in SPLITS],
    split_summary["gambar"],
    color=[SPLIT_COLORS[split] for split in SPLITS],
)
ax.bar_label(bars, padding=3)
ax.set(title="Jumlah gambar pada setiap partisi", ylabel="Jumlah gambar (file)")
ax.margins(y=0.15)
finish_figure(
    fig,
    "eda_01_split_distribution",
    "Keanggotaan gambar unik sesuai sumber YAML; instance dilaporkan terpisah.",
)
interpret(
    "EDA-01",
    "; ".join(
        f"{SPLIT_LABELS[row.split]}: {row.gambar:,} gambar "
        f"({row.proporsi_gambar_pct:.2f}%) dan {row.instance_valid:,} instance valid"
        for row in split_summary.itertuples()
    )
    + ". "
    + (
        "Test tersedia."
        if "test" in SPLITS
        else "Test tidak tercantum; tidak dibentuk split baru."
    ),
)


## 5. Distribusi kelas objek beranotasi

### 5.1 Instance, gambar per kelas, dan perbedaan partisi — EDA-02/03

**Tujuan:** Mengidentifikasi komposisi kelas dengan membedakan instance dan gambar yang memuat kelas.

**Metode:** Instance adalah jumlah baris bbox valid. Gambar per kelas dihitung dengan nunique(image_id), sehingga beberapa objek kelas sama dalam satu gambar hanya menambah satu gambar. Urutan kelas mengikuti ID YAML dan kelas nol instance tetap ditampilkan.

**Caption:** EDA-02 menampilkan instance keseluruhan; EDA-03 membandingkan instance per kelas dan partisi.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Perbedaan komposisi membantu memberi konteks precision, recall, dan AP per kelas pada evaluasi terpisah.

In [ ]:
class_rows = []
for split in ["seluruh", *SPLITS]:
    part = boxes_df if split == "seluruh" else boxes_df.loc[boxes_df["split"].eq(split)]
    n_images = (
        len(images_df)
        if split == "seluruh"
        else int(images_df["split"].eq(split).sum())
    )
    for class_id in CLASS_IDS:
        selected = part.loc[part["class_id"].eq(class_id)]
        class_rows.append(
            {
                "split": split,
                "class_id": class_id,
                "kelas": CLASS_NAMES[class_id],
                "instance": len(selected),
                "gambar_memuat_kelas": selected["image_id"].nunique(),
                "proporsi_instance_pct": 100 * len(selected) / len(part)
                if len(part)
                else np.nan,
                "proporsi_gambar_pct": 100 * selected["image_id"].nunique() / n_images
                if n_images
                else np.nan,
            }
        )
class_distribution = register_table("class_distribution", pd.DataFrame(class_rows))
class_total = class_distribution.loc[class_distribution["split"].eq("seluruh")].copy()
fig, ax = plt.subplots(figsize=(max(7, len(CLASS_NAMES) * 0.8), 4.5))
bars = ax.bar(
    CLASS_NAMES, class_total["instance"], color=[CLASS_COLORS[i] for i in CLASS_IDS]
)
ax.bar_label(bars, padding=3)
ax.set(title="Distribusi instance per kelas", ylabel="Jumlah bbox valid (instance)")
ax.tick_params(axis="x", labelrotation=30)
ax.margins(y=0.18)
finish_figure(
    fig,
    "eda_02_class_distribution",
    "Jumlah baris bbox valid; kotak identik valid tetap dihitung.",
)
fig, ax = plt.subplots(figsize=(max(8, len(CLASS_NAMES)), 4.5))
positions = np.arange(len(CLASS_NAMES))
bar_width = 0.8 / len(SPLITS)
for index, split in enumerate(SPLITS):
    counts = class_distribution.loc[
        class_distribution["split"].eq(split), "instance"
    ].to_numpy()
    ax.bar(
        positions + (index - (len(SPLITS) - 1) / 2) * bar_width,
        counts,
        width=bar_width,
        label=SPLIT_LABELS[split],
        color=SPLIT_COLORS[split],
    )
ax.set_xticks(positions, CLASS_NAMES, rotation=30, ha="right")
ax.set(
    title="Distribusi kelas pada setiap partisi", ylabel="Jumlah bbox valid (instance)"
)
ax.legend(title="Partisi")
finish_figure(
    fig,
    "eda_03_class_by_split",
    "Instance per kelas dibandingkan dengan denominator split masing-masing.",
)
if len(boxes_df):
    dominant = class_total.loc[class_total["instance"].idxmax()]
    interpret(
        "EDA-02/03",
        f"Kelas dengan instance terbanyak adalah {dominant['kelas']} "
        f"({dominant['instance']:,} instance; {dominant['proporsi_instance_pct']:.2f}%), "
        f"terdapat pada {dominant['gambar_memuat_kelas']:,} gambar. "
        "Jumlah gambar antarkelas tidak dapat dijumlahkan sebagai jumlah gambar dataset karena ko-kemunculan.",
    )
else:
    interpret(
        "EDA-02/03",
        "Tidak ada bbox valid yang memenuhi syarat; komposisi kelas belum dapat ditafsirkan.",
    )


### 5.2 Proporsi dan rasio ketidakseimbangan — EDA-04

**Tujuan:** Mendokumentasikan besar ketidakseimbangan dan kelas tanpa instance.

**Metode:** Rasio global = jumlah kelas terbesar/jumlah kelas terkecil. Bila kelas minimum nol dan ada instance, rasio seluruh kelas tak terhingga; rasio pada kelas hadir dilaporkan terpisah. Bila seluruh kelas nol, rasio tidak terdefinisi. Proporsi gambar dapat kurang dari kondisi sebenarnya jika label hilang atau sebagian tidak valid.

**Caption:** Proporsi instance dan gambar per kelas serta rasio seluruh kelas/kelas hadir pada setiap split.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Ketidakseimbangan perlu dipertimbangkan saat membahas metrik per kelas; EDA tidak menetapkan strategi training atau bobot KD.

In [ ]:
imbalance_rows = []
for split in ["seluruh", *SPLITS]:
    counts = class_distribution.loc[class_distribution["split"].eq(split), "instance"]
    positive = counts[counts > 0]
    ratio_all = (
        np.nan
        if positive.empty
        else np.inf
        if counts.min() == 0
        else counts.max() / counts.min()
    )
    ratio_present = positive.max() / positive.min() if len(positive) else np.nan
    imbalance_rows.append(
        {
            "split": split,
            "instance_terbesar": counts.max(),
            "instance_terkecil": counts.min(),
            "kelas_tanpa_instance": int(counts.eq(0).sum()),
            "imbalance_ratio_semua_kelas": ratio_all,
            "imbalance_ratio_kelas_hadir": ratio_present,
        }
    )
imbalance_table = register_table("class_imbalance", pd.DataFrame(imbalance_rows))
register_table(
    "class_proportions",
    class_distribution[
        [
            "split",
            "class_id",
            "kelas",
            "instance",
            "gambar_memuat_kelas",
            "proporsi_instance_pct",
            "proporsi_gambar_pct",
        ]
    ],
)
overall_imbalance = imbalance_table.iloc[0]
interpret(
    "EDA-04",
    f"Terdapat {int(overall_imbalance['kelas_tanpa_instance'])} kelas tanpa instance valid. "
    f"Rasio seluruh kelas = {overall_imbalance['imbalance_ratio_semua_kelas']:.3g}; "
    f"rasio kelas hadir = {overall_imbalance['imbalance_ratio_kelas_hadir']:.3g}. "
    "Nilai inf menandakan denominator nol; NaN menandakan tidak terdefinisi.",
)


## 6. Distribusi jumlah objek per gambar

### 6.1 Kepadatan anotasi — EDA-05

**Tujuan:** Menggambarkan banyaknya objek beranotasi per gambar, termasuk background yang diketahui.

**Metode:** Jumlah objek memakai seluruh baris valid pada gambar dengan label lengkap dan pasangan unik. Label hilang, tidak terbaca, atau memuat baris invalid dikeluarkan dari distribusi ini; jumlah baris valid parsial tetap tersedia pada tabel mentah. ECDF F(x)=jumlah gambar dengan count≤x/denominator gambar yang memenuhi syarat.

**Caption:** Histogram, boxplot, dan ECDF jumlah objek untuk setiap split; tabel deskriptif mencantumkan denominator.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Kepadatan anotasi memberi konteks kerumitan citra, tanpa menyatakan bahwa banyak objek menyebabkan penurunan performa model.

In [ ]:
register_table(
    "object_count_statistics", describe_numeric(count_images_df, ["n_objects"], "split")
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
maximum = int(count_images_df["n_objects"].max()) if len(count_images_df) else 0
bins = np.arange(-0.5, maximum + 1.5) if maximum <= 60 else 40
box_data, box_labels, box_colors = [], [], []
for split in SPLITS:
    values = count_images_df.loc[
        count_images_df["split"].eq(split), "n_objects"
    ].dropna()
    if len(values):
        axes[0].hist(
            values,
            bins=bins,
            histtype="step",
            linewidth=1.8,
            label=SPLIT_LABELS[split],
            color=SPLIT_COLORS[split],
        )
        sorted_values = np.sort(values.to_numpy())
        x = np.r_[sorted_values[0] - 0.5, sorted_values]
        y = np.r_[0, np.arange(1, len(values) + 1) / len(values)]
        axes[2].step(
            x, y, where="post", label=SPLIT_LABELS[split], color=SPLIT_COLORS[split]
        )
        box_data.append(values)
        box_labels.append(SPLIT_LABELS[split])
        box_colors.append(SPLIT_COLORS[split])
if box_data:
    plotted = axes[1].boxplot(box_data, tick_labels=box_labels, patch_artist=True)
    for patch, color in zip(plotted["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[0].legend(title="Partisi")
    axes[2].legend(title="Partisi")
else:
    for ax in axes:
        empty_axis(ax)
axes[0].set(
    title="Frekuensi jumlah objek",
    xlabel="Objek per gambar (instance)",
    ylabel="Jumlah gambar (file)",
)
axes[1].set(title="Sebaran jumlah objek", ylabel="Objek per gambar (instance)")
axes[2].set(
    title="Distribusi kumulatif empiris",
    xlabel="Objek per gambar (instance)",
    ylabel="Proporsi kumulatif gambar",
)
axes[2].set_ylim(0, 1.02)
finish_figure(
    fig,
    "eda_05_object_count",
    "Distribusi menggunakan gambar dengan label lengkap; jumlah objek beranotasi bukan estimasi semua objek nyata.",
)
interpret(
    "EDA-05",
    f"Denominator {len(count_images_df):,}/{len(images_df):,} gambar. "
    + (
        f"Median {count_images_df['n_objects'].median():.2f}, P95 "
        f"{count_images_df['n_objects'].quantile(0.95):.2f}, maksimum "
        f"{count_images_df['n_objects'].max():.0f} instance per gambar."
        if len(count_images_df)
        else "Tidak tersedia gambar dengan label lengkap."
    ),
)


## 7. Karakteristik citra

### 7.1 Resolusi, aspect ratio citra, dan ukuran file — EDA-06

**Tujuan:** Memeriksa variasi resolusi dan dimensi fisik sumber data.

**Metode:** Lebar/tinggi berasal dari metadata gambar yang lolos pemeriksaan integritas. Aspect ratio citra = W/H. Ukuran file adalah byte/1024² dalam MiB, bukan ukuran tensor. Tabel deskriptif ukuran file mencakup semua file dengan metadata byte tersedia; histogram hanya citra terverifikasi. Tabel frekuensi resolusi dan deskriptif tersedia untuk setiap split.

**Caption:** Scatter resolusi, histogram aspect ratio, dan histogram ukuran file sumber.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Resolusi sumber perlu dibedakan dari input eksperimen 640 dan transformasi letterbox; EDA tidak mensimulasikan augmentasi training.

In [ ]:
readable_images = images_df.loc[images_df["readable"]].copy()
images_df["file_mib"] = images_df["file_bytes"] / (1024**2)
readable_images["file_mib"] = readable_images["file_bytes"] / (1024**2)
register_table(
    "resolution_frequency",
    readable_images.groupby(["split", "width_px", "height_px"])
    .size()
    .reset_index(name="jumlah_gambar"),
)
register_table(
    "image_statistics",
    describe_numeric(
        images_df, ["width_px", "height_px", "image_aspect_ratio", "file_mib"], "split"
    ),
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ratio_bins = np.histogram_bin_edges(readable_images["image_aspect_ratio"], bins=30)
file_bins = np.histogram_bin_edges(readable_images["file_mib"], bins=30)
for split in SPLITS:
    part = readable_images.loc[readable_images["split"].eq(split)]
    if len(part):
        axes[0].scatter(
            part["width_px"],
            part["height_px"],
            s=18,
            alpha=0.45,
            color=SPLIT_COLORS[split],
            label=SPLIT_LABELS[split],
        )
        axes[1].hist(
            part["image_aspect_ratio"],
            bins=ratio_bins,
            histtype="step",
            linewidth=1.5,
            color=SPLIT_COLORS[split],
            label=SPLIT_LABELS[split],
        )
        axes[2].hist(
            part["file_mib"],
            bins=file_bins,
            histtype="step",
            linewidth=1.5,
            color=SPLIT_COLORS[split],
            label=SPLIT_LABELS[split],
        )
if readable_images.empty:
    for ax in axes:
        empty_axis(ax)
else:
    axes[0].legend(title="Partisi")
axes[0].set(
    title="Resolusi gambar sumber",
    xlabel="Lebar gambar (piksel)",
    ylabel="Tinggi gambar (piksel)",
)
axes[1].set(
    title="Aspect ratio citra",
    xlabel="Rasio lebar/tinggi",
    ylabel="Jumlah gambar (file)",
)
axes[2].set(
    title="Ukuran file citra", xlabel="Ukuran file (MiB)", ylabel="Jumlah gambar (file)"
)
finish_figure(
    fig,
    "eda_06_image_characteristics",
    "Resolusi pixel mentah, tanpa resizing atau rotasi EXIF otomatis.",
)
interpret(
    "EDA-06",
    f"{len(readable_images):,} gambar lolos pemeriksaan {EDA_MODE}; "
    f"terdapat {len(readable_images[['width_px', 'height_px']].drop_duplicates()):,} kombinasi resolusi. "
    "Ukuran file terkompresi tidak ekuivalen dengan kompleksitas visual atau kualitas anotasi.",
)


### 7.2 Kecerahan citra — EDA-07

**Tujuan:** Mendeskripsikan variasi intensitas kecerahan yang teramati.

**Metode:** Brightness = rata-rata gray level Pillow convert('L') berskala 0–255 pada thumbnail dengan sisi maksimum terkonfigurasi, tanpa autocontrast. Standard mengambil sampel tanpa pengembalian maksimal 500 gambar terverifikasi per split memakai seed tetap; full memakai semua gambar terverifikasi. Estimator thumbnail adalah proksi intensitas, bukan pengukuran iluminasi fisik.

**Caption:** Histogram brightness per split dan tabel cakupan sampel/keberhasilan decoding.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Deskripsi kecerahan memberi konteks variasi citra; tidak membuktikan kondisi malam/hujan dan tidak menjadi dasar tuning dari test.

In [ ]:
def image_brightness(path: Path, thumbnail_side: int) -> float:
    """Rata-rata grayscale thumbnail; satu citra didekode pada satu waktu."""
    with Image.open(path) as image:
        image = image.convert("RGB")
        image.thumbnail((thumbnail_side, thumbnail_side), Image.Resampling.LANCZOS)
        return float(np.asarray(image.convert("L"), dtype=np.float32).mean())


brightness_started = time.perf_counter()
brightness_coverage = []
rng = np.random.default_rng(SEED)
for split in SPLITS:
    eligible = images_df.index[
        images_df["split"].eq(split) & images_df["readable"]
    ].to_numpy()
    count = (
        len(eligible)
        if EDA_MODE == "full"
        else min(len(eligible), BRIGHTNESS_SAMPLE_PER_SPLIT)
    )
    selected = np.sort(rng.choice(eligible, count, replace=False))
    succeeded = 0
    for index in selected:
        images_df.at[index, "brightness_sampled"] = True
        path = Path(images_df.at[index, "image_path"])
        try:
            images_df.at[index, "brightness"] = image_brightness(
                path, BRIGHTNESS_THUMBNAIL_SIDE
            )
            succeeded += 1
        except (OSError, ValueError, Image.DecompressionBombError) as error:
            add_issue(split, path, "decode_brightness_gagal", "CRITICAL", str(error))
    brightness_coverage.append(
        {
            "split": split,
            "gambar_terinventarisasi": int(images_df["split"].eq(split).sum()),
            "gambar_lolos_verifikasi": len(eligible),
            "dipilih": count,
            "berhasil": succeeded,
            "gagal": count - succeeded,
            "proporsi_terverifikasi_pct": 100 * count / len(eligible)
            if len(eligible)
            else np.nan,
        }
    )
TIMINGS["brightness"] = time.perf_counter() - brightness_started
register_table("brightness_coverage", pd.DataFrame(brightness_coverage))
brightness_df = images_df.loc[images_df["brightness"].notna()].copy()
register_table(
    "brightness_statistics", describe_numeric(brightness_df, ["brightness"], "split")
)
fig, axes = plt.subplots(1, len(SPLITS), figsize=(5 * len(SPLITS), 4), squeeze=False)
for ax, split in zip(axes.flat, SPLITS):
    values = brightness_df.loc[brightness_df["split"].eq(split), "brightness"]
    if len(values):
        ax.hist(
            values, bins=np.linspace(0, 255, 31), color=SPLIT_COLORS[split], alpha=0.8
        )
    else:
        empty_axis(ax)
    ax.set(
        title=f"{SPLIT_LABELS[split]} (n={len(values):,})",
        xlabel="Rata-rata grayscale (0–255)",
        ylabel="Jumlah gambar yang diukur (file)",
        xlim=(0, 255),
    )
finish_figure(
    fig,
    "eda_07_brightness",
    "Cakupan dan metode sampling dinyatakan; frekuensi sampel tidak dianggap frekuensi seluruh dataset.",
)
interpret(
    "EDA-07",
    f"Brightness berhasil diukur pada {len(brightness_df):,}/{len(images_df):,} gambar; "
    f"mode {EDA_MODE}, seed {SEED}, thumbnail maksimum {BRIGHTNESS_THUMBNAIL_SIDE} piksel. "
    + (
        f"Median gabungan gambar yang diukur {brightness_df['brightness'].median():.2f}/255. "
        if len(brightness_df)
        else ""
    )
    + "Statistik gabungan sampel tidak diberi bobot ulang untuk mengestimasi populasi.",
)


## 8. Karakteristik bounding box

### 8.1 Dimensi dan bentuk bounding box — EDA-08

**Tujuan:** Mendeskripsikan variasi lebar/tinggi kotak dan membedakan normalisasi dari ukuran piksel.

**Metode:** Lebar relatif=w dan tinggi relatif=h pada anotasi YOLO. Dimensi pixel=(wW,hH), luas relatif=w×h, luas pixel=(wW)(hH). Aspect ratio pixel=(wW)/(hH); w/h hanya rasio dalam koordinat normalisasi dan berbeda pada gambar tidak persegi. Statistik per split dan kelas memakai seluruh bbox valid dari pasangan unik.

**Caption:** Scatter lebar versus tinggi ternormalisasi dengan warna kelas serta tabel deskriptif geometri.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Ukuran dan bentuk objek dapat memberi konteks kemampuan representasi teacher/student; EDA tidak mengukur kemampuan model.

In [ ]:
bbox_columns = [
    "width_norm",
    "height_norm",
    "width_box_px",
    "height_box_px",
    "area_relative",
    "area_px",
    "aspect_ratio_pixel",
    "aspect_ratio_normalized",
    "x_center",
    "y_center",
]
register_table(
    "bounding_box_statistics", describe_numeric(boxes_df, bbox_columns, "split")
)
register_table(
    "bounding_box_statistics_by_class",
    describe_numeric(boxes_df, bbox_columns, "class_name"),
)
fig, ax = plt.subplots(figsize=(8, 6))
for class_id in CLASS_IDS:
    part = boxes_df.loc[boxes_df["class_id"].eq(class_id)]
    if len(part):
        ax.scatter(
            part["width_norm"],
            part["height_norm"],
            s=8,
            alpha=0.25,
            color=CLASS_COLORS[class_id],
            label=CLASS_NAMES[class_id],
            rasterized=True,
        )
if boxes_df.empty:
    empty_axis(ax)
else:
    ax.legend(title="Kelas", bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=2)
ax.set(
    title="Dimensi bounding box ternormalisasi",
    xlabel="Lebar bbox/lebar citra (0–1)",
    ylabel="Tinggi bbox/tinggi citra (0–1)",
    xlim=(0, 1),
    ylim=(0, 1),
)
finish_figure(
    fig,
    "eda_08_bbox_dimensions",
    "Setiap titik adalah satu bbox valid; titik yang bertumpuk dapat menutupi frekuensi.",
)
interpret(
    "EDA-08",
    f"Geometri relatif tersedia untuk {len(boxes_df):,} bbox; geometri pixel tersedia "
    f"untuk {boxes_df['area_px'].notna().sum():,} bbox. "
    "Kotak pada gambar gagal verifikasi tidak mempunyai statistik pixel.",
)


### 8.2 Luas relatif dan aspect ratio per kelas — EDA-09/10

**Tujuan:** Mengidentifikasi variasi skala relatif serta bentuk objek setiap kelas.

**Metode:** Luas relatif = w×h terhadap luas gambar asli. Histogram memakai bin bersama setiap split. Aspect ratio menggunakan dimensi pixel asli, disajikan pada sumbu x log untuk rentang rasio lebar/tinggi; bin juga logaritmik. Nilai ekstrem tidak dipotong.

**Caption:** EDA-09 memuat histogram dan boxplot luas relatif; EDA-10 histogram aspect ratio pixel per kelas.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Distribusi membantu mendokumentasikan objek yang menempati bagian kecil citra; tidak otomatis menyatakan kategori COCO atau kualitas deteksi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
area_data, area_labels = [], []
for split in SPLITS:
    values = boxes_df.loc[boxes_df["split"].eq(split), "area_relative"]
    if len(values):
        axes[0].hist(
            values,
            bins=np.linspace(0, 1, 51),
            histtype="step",
            linewidth=1.8,
            color=SPLIT_COLORS[split],
            label=SPLIT_LABELS[split],
        )
        area_data.append(values)
        area_labels.append(SPLIT_LABELS[split])
if area_data:
    axes[0].legend(title="Partisi")
    axes[1].boxplot(area_data, tick_labels=area_labels)
else:
    for ax in axes:
        empty_axis(ax)
axes[0].set(
    title="Luas relatif bbox",
    xlabel="Luas bbox/luas citra (0–1)",
    ylabel="Jumlah bbox (instance)",
)
axes[1].set(title="Sebaran luas relatif", ylabel="Luas bbox/luas citra (0–1)")
finish_figure(
    fig,
    "eda_09_relative_area",
    "Luas relatif dihitung pada citra asli tanpa transformasi input model.",
)
fig, axes = plt.subplots(
    math.ceil(len(CLASS_NAMES) / 3),
    min(3, len(CLASS_NAMES)),
    figsize=(14, 3.8 * math.ceil(len(CLASS_NAMES) / 3)),
    squeeze=False,
)
ratio_values = (
    boxes_df["aspect_ratio_pixel"].replace([np.inf, -np.inf], np.nan).dropna()
)
if len(ratio_values):
    lower, upper = ratio_values.min(), ratio_values.max()
    if lower == upper:
        lower, upper = lower / 1.1, upper * 1.1
    ratio_bins = np.geomspace(lower, upper, 31)
for class_id, ax in zip(CLASS_IDS, axes.flat):
    values = boxes_df.loc[
        boxes_df["class_id"].eq(class_id), "aspect_ratio_pixel"
    ].dropna()
    if len(values):
        ax.hist(values, bins=ratio_bins, color=CLASS_COLORS[class_id], alpha=0.85)
        ax.set_xscale("log")
    else:
        empty_axis(ax)
    ax.set(
        title=CLASS_NAMES[class_id],
        xlabel="Rasio lebar/tinggi pixel (sumbu log)",
        ylabel="Jumlah bbox (instance)",
    )
for ax in list(axes.flat)[len(CLASS_NAMES) :]:
    ax.set_visible(False)
finish_figure(
    fig,
    "eda_10_bbox_aspect_ratio",
    "Bin aspect ratio logaritmik sama untuk seluruh kelas; y tetap frekuensi instance.",
)
interpret(
    "EDA-09/10",
    (
        f"Median luas relatif {boxes_df['area_relative'].median():.4f}; "
        f"median aspect ratio pixel {ratio_values.median():.3f}. "
        if len(boxes_df)
        else "Tidak ada bbox valid. "
    )
    + "Aspect ratio pixel dibedakan dari rasio koordinat normalisasi.",
)


### 8.3 Ukuran relatif objek per kelas — EDA-12

**Tujuan:** Membandingkan luas relatif objek antarkelas dan menyediakan kategori operasional transparan.

**Metode:** Boxplot memakai area relatif asli. Kategori kecil relatif: area<t1; sedang relatif: t1≤area<t2; besar relatif: area≥t2. Ambang t1/t2 berasal dari konfigurasi. Kategori ini bukan definisi COCO dan tidak setara dengan area pixel pada input 640 setelah resizing/letterbox.

**Caption:** Boxplot luas relatif per kelas serta tabel jumlah kategori dan denominator instance.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Kategori membantu pembahasan konteks variasi objek tanpa digunakan untuk mengubah anotasi atau protokol eksperimen.

In [ ]:
size_table = pd.crosstab(
    boxes_df["class_id"], boxes_df["size_relative"], dropna=False
).reindex(
    index=CLASS_IDS,
    columns=["Kecil relatif", "Sedang relatif", "Besar relatif"],
    fill_value=0,
)
size_table.index = pd.Index(CLASS_NAMES, name="kelas")
register_table("relative_size_by_class", size_table.reset_index())
fig, ax = plt.subplots(figsize=(max(8, len(CLASS_NAMES)), 5))
nonempty_ids = [
    class_id for class_id in CLASS_IDS if boxes_df["class_id"].eq(class_id).any()
]
if nonempty_ids:
    data = [
        boxes_df.loc[boxes_df["class_id"].eq(class_id), "area_relative"]
        for class_id in nonempty_ids
    ]
    plotted = ax.boxplot(data, positions=nonempty_ids, patch_artist=True)
    for patch, class_id in zip(plotted["boxes"], nonempty_ids):
        patch.set_facecolor(CLASS_COLORS[class_id])
        patch.set_alpha(0.6)
else:
    empty_axis(ax)
ax.set_xticks(CLASS_IDS, CLASS_NAMES, rotation=30, ha="right")
ax.set(title="Luas relatif bbox menurut kelas", ylabel="Luas bbox/luas citra (0–1)")
finish_figure(
    fig,
    "eda_12_size_by_class",
    f"Kategori operasional: kecil <{small_threshold:g}; sedang [{small_threshold:g}, {medium_threshold:g}); besar ≥{medium_threshold:g}.",
)
interpret(
    "EDA-12",
    f"Ambang luas relatif {small_threshold:g} dan {medium_threshold:g}; "
    f"jumlah bbox kecil relatif {int((boxes_df['area_relative'] < small_threshold).sum()):,}. "
    "Tidak digunakan label skala COCO; efek input 640 tidak dihitung sebagai ukuran asli.",
)


## 9. Distribusi spasial objek

### 9.1 Heatmap pusat bounding box — EDA-11

**Tujuan:** Mengidentifikasi wilayah citra yang lebih sering memuat anotasi.

**Metode:** Histogram dua dimensi pusat (x_center,y_center) pada grid [0,1]². Nilai setiap bin adalah jumlah bbox, bukan jumlah gambar. Semua kelas memakai batas grid dan skala warna per kelas yang sama. Koordinat y=0 berada di bagian atas gambar. Heatmap gabungan dan per kelas tersedia.

**Caption:** Heatmap frekuensi pusat bbox gabungan dan masing-masing kelas.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Pola spasial dapat merefleksikan komposisi sudut kamera dan lalu lintas; tidak membuktikan bias model.

In [ ]:
spatial_edges = np.linspace(0, 1, SPATIAL_BINS + 1)
spatial_rows = []
grids = []
for class_id in CLASS_IDS:
    part = boxes_df.loc[boxes_df["class_id"].eq(class_id)]
    grid, _, _ = np.histogram2d(
        part["y_center"], part["x_center"], bins=[spatial_edges, spatial_edges]
    )
    grids.append(grid)
    for row in range(SPATIAL_BINS):
        for column in range(SPATIAL_BINS):
            spatial_rows.append(
                {
                    "class_id": class_id,
                    "kelas": CLASS_NAMES[class_id],
                    "x_bin": column,
                    "y_bin": row,
                    "x_min": spatial_edges[column],
                    "x_max": spatial_edges[column + 1],
                    "y_min": spatial_edges[row],
                    "y_max": spatial_edges[row + 1],
                    "instance": int(grid[row, column]),
                }
            )
register_table("spatial_bins", pd.DataFrame(spatial_rows), show=False)
fig, ax = plt.subplots(figsize=(7, 5.5))
combined_grid = np.sum(grids, axis=0)
image_plot = ax.imshow(
    combined_grid, extent=[0, 1, 1, 0], origin="upper", cmap="viridis", vmin=0
)
fig.colorbar(image_plot, ax=ax, label="Jumlah bbox per bin (instance)")
ax.set(
    title="Distribusi pusat bbox seluruh kelas",
    xlabel="Pusat horizontal x (0–1)",
    ylabel="Pusat vertikal y (0–1)",
)
finish_figure(
    fig,
    "eda_11_spatial_centers",
    "y=0 adalah bagian atas; setiap bin menjumlahkan instance pada seluruh partisi tersedia.",
)
fig, axes = plt.subplots(
    math.ceil(len(CLASS_NAMES) / 3),
    min(3, len(CLASS_NAMES)),
    figsize=(14, 4 * math.ceil(len(CLASS_NAMES) / 3)),
    squeeze=False,
)
shared_maximum = max(1, max(float(grid.max()) for grid in grids))
for class_id, ax in zip(CLASS_IDS, axes.flat):
    plotted = ax.imshow(
        grids[class_id],
        extent=[0, 1, 1, 0],
        origin="upper",
        cmap="viridis",
        vmin=0,
        vmax=shared_maximum,
    )
    ax.set(title=CLASS_NAMES[class_id], xlabel="Pusat x (0–1)", ylabel="Pusat y (0–1)")
    fig.colorbar(plotted, ax=ax, label="BBox/bin (instance)", shrink=0.8)
for ax in list(axes.flat)[len(CLASS_NAMES) :]:
    ax.set_visible(False)
finish_figure(
    fig,
    "eda_11_spatial_centers_by_class",
    "Skala frekuensi dan bin konsisten; heatmap bukan density map target supervised.",
)
interpret(
    "EDA-11",
    f"Grid {SPATIAL_BINS}×{SPATIAL_BINS} memuat {int(combined_grid.sum()):,} pusat bbox. "
    "Akumulasi instance dapat didominasi kelas/split berukuran besar; pola gabungan tidak mewakili tiap kelas secara sama.",
)


## 10. Kepadatan dan kedekatan objek

### 10.1 Jumlah objek versus ukuran relatif — EDA-13

**Tujuan:** Mendeskripsikan hubungan jumlah anotasi dengan ukuran objek dalam gambar yang sama.

**Metode:** Agregasi per gambar dengan label lengkap: median area relatif bbox dan jumlah bbox. Gambar background tanpa bbox tidak mempunyai median area dan dikeluarkan dari scatter. Spearman adalah korelasi peringkat deskriptif, tanpa pengujian kausal.

**Caption:** Scatter jumlah objek versus median area relatif per gambar, dengan warna partisi.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Hubungan teramati membantu merumuskan konteks analisis KD, tanpa pemilihan hyperparameter dari test.

In [ ]:
area_per_image = (
    boxes_df.groupby("image_id")["area_relative"]
    .agg(
        mean_area_relative="mean",
        median_area_relative="median",
        sum_area_relative="sum",
    )
    .reset_index()
)
density_df = count_images_df[["image_id", "split", "n_objects"]].merge(
    area_per_image, on="image_id", how="left", validate="one_to_one"
)
register_table("density_per_image", density_df, show=False)
register_table(
    "density_statistics",
    describe_numeric(
        density_df,
        [
            "n_objects",
            "mean_area_relative",
            "median_area_relative",
            "sum_area_relative",
        ],
        "split",
    ),
)
fig, ax = plt.subplots(figsize=(8, 5))
for split in SPLITS:
    part = density_df.loc[density_df["split"].eq(split)].dropna(
        subset=["median_area_relative"]
    )
    if len(part):
        ax.scatter(
            part["n_objects"],
            part["median_area_relative"],
            s=16,
            alpha=0.4,
            color=SPLIT_COLORS[split],
            label=SPLIT_LABELS[split],
        )
if density_df["median_area_relative"].notna().any():
    ax.legend(title="Partisi")
else:
    empty_axis(ax)
ax.set(
    title="Jumlah anotasi dan ukuran relatif objek",
    xlabel="Objek per gambar (instance)",
    ylabel="Median luas bbox/luas citra",
)
finish_figure(
    fig,
    "eda_13_count_vs_area",
    "Satu titik per gambar berlabel lengkap yang memuat bbox; jumlah luas bukan luas union dan dapat melebihi 1.",
)
correlation_rows = []
for split in ["seluruh", *SPLITS]:
    part = (
        density_df
        if split == "seluruh"
        else density_df.loc[density_df["split"].eq(split)]
    )
    part = part.dropna(subset=["n_objects", "median_area_relative"])
    correlation = (
        part["n_objects"].rank().corr(part["median_area_relative"].rank())
        if len(part) >= 2
        and part["n_objects"].nunique() > 1
        and part["median_area_relative"].nunique() > 1
        else np.nan
    )
    correlation_rows.append(
        {"split": split, "gambar": len(part), "spearman_count_median_area": correlation}
    )
register_table("count_area_correlation", pd.DataFrame(correlation_rows))
interpret(
    "EDA-13",
    f"Scatter memuat {density_df['median_area_relative'].notna().sum():,} gambar. "
    f"Spearman gabungan {correlation_rows[0]['spearman_count_median_area']:.3f}; "
    "nilai NaN menandakan variasi atau jumlah observasi tidak mencukupi. Korelasi gabungan dapat dipengaruhi komposisi split.",
)


### 10.2 IoU dan jarak pusat terdekat

**Tujuan:** Mengukur kedekatan geometris bbox sebagai proksi kepadatan spasial.

**Metode:** IoU pasangan = luas irisan/(luas A+luas B−irisan); pasangan dihitung sekali pada segitiga atas. Jarak terdekat per bbox memakai Euclidean pusat pada koordinat [0,1]², bersifat anisotropik pada citra tidak persegi dan bukan jarak fisik jalan. Semua pasangan diproses bila 2≤jumlah objek≤batas konfigurasi; gambar di atas batas dilewati dan dicatat. Nilai n<2 tidak terdefinisi.

**Caption:** Histogram IoU maksimum per gambar dan jarak terdekat per bbox; tabel cakupan, denominator pasangan, dan statistik.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Overlap bbox tidak membuktikan occlusion piksel atau kesalahan anotasi; kedekatan tidak membuktikan kemiripan objek.

In [ ]:
def pairwise_geometry(boxes: pd.DataFrame) -> tuple[dict, np.ndarray]:
    """Hitung semua pasangan dalam satu gambar yang berada di bawah batas biaya."""
    n = len(boxes)
    if n < 2:
        return {
            "pair_count": 0,
            "overlap_pairs": 0,
            "max_iou": np.nan,
            "mean_iou": np.nan,
            "overlap_fraction": np.nan,
        }, np.full(n, np.nan)
    centers = boxes[["x_center", "y_center"]].to_numpy(dtype=float)
    sizes = boxes[["width_norm", "height_norm"]].to_numpy(dtype=float)
    lower, upper = centers - sizes / 2, centers + sizes / 2
    intersection_sizes = np.maximum(
        0,
        np.minimum(upper[:, None, :], upper[None, :, :])
        - np.maximum(lower[:, None, :], lower[None, :, :]),
    )
    intersection = intersection_sizes.prod(axis=2)
    areas = sizes.prod(axis=1)
    iou = intersection / (areas[:, None] + areas[None, :] - intersection)
    upper_pairs = iou[np.triu_indices(n, k=1)]
    distances = np.linalg.norm(centers[:, None, :] - centers[None, :, :], axis=2)
    np.fill_diagonal(distances, np.inf)
    return {
        "pair_count": len(upper_pairs),
        "overlap_pairs": int((upper_pairs > 0).sum()),
        "max_iou": float(upper_pairs.max()),
        "mean_iou": float(upper_pairs.mean()),
        "overlap_fraction": float((upper_pairs > 0).mean()),
    }, distances.min(axis=1)


pair_started = time.perf_counter()
pair_rows, neighbor_rows = [], []
box_groups = {key: part for key, part in boxes_df.groupby("image_id", sort=False)}
for image in count_images_df.itertuples():
    n = int(image.n_objects)
    status = (
        "kurang dari dua objek"
        if n < 2
        else "di atas batas biaya"
        if n > PAIRWISE_MAX_OBJECTS
        else "diperiksa"
    )
    row = {
        "image_id": image.image_id,
        "split": image.split,
        "n_objects": n,
        "status": status,
        "pair_count": 0,
        "overlap_pairs": 0,
        "max_iou": np.nan,
        "mean_iou": np.nan,
        "overlap_fraction": np.nan,
    }
    if status == "diperiksa":
        part = box_groups[image.image_id]
        metrics, nearest = pairwise_geometry(part)
        row.update(metrics)
        neighbor_rows.extend(
            {
                "image_id": image.image_id,
                "split": image.split,
                "line": int(line),
                "nearest_center_distance": float(distance),
            }
            for line, distance in zip(part["line"], nearest)
        )
    pair_rows.append(row)
pair_df = pd.DataFrame(
    pair_rows,
    columns=[
        "image_id",
        "split",
        "n_objects",
        "status",
        "pair_count",
        "overlap_pairs",
        "max_iou",
        "mean_iou",
        "overlap_fraction",
    ],
)
nearest_df = pd.DataFrame(
    neighbor_rows, columns=["image_id", "split", "line", "nearest_center_distance"]
)
TIMINGS["geometri_pasangan"] = time.perf_counter() - pair_started
register_table("overlap_per_image", pair_df, show=False)
register_table("nearest_centers_per_box", nearest_df, show=False)
register_table(
    "pairwise_coverage",
    pair_df.groupby(["split", "status"]).size().reset_index(name="gambar"),
)
register_table(
    "overlap_statistics",
    describe_numeric(pair_df, ["max_iou", "mean_iou", "overlap_fraction"], "split"),
)
register_table(
    "nearest_center_statistics",
    describe_numeric(nearest_df, ["nearest_center_distance"], "split"),
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split in SPLITS:
    iou_values = pair_df.loc[pair_df["split"].eq(split), "max_iou"].dropna()
    distances = nearest_df.loc[nearest_df["split"].eq(split), "nearest_center_distance"]
    if len(iou_values):
        axes[0].hist(
            iou_values,
            bins=np.linspace(0, 1, 31),
            histtype="step",
            linewidth=1.5,
            label=SPLIT_LABELS[split],
            color=SPLIT_COLORS[split],
        )
        axes[1].hist(
            distances,
            bins=np.linspace(0, np.sqrt(2), 31),
            histtype="step",
            linewidth=1.5,
            label=SPLIT_LABELS[split],
            color=SPLIT_COLORS[split],
        )
if pair_df["max_iou"].notna().any():
    axes[0].legend(title="Partisi")
else:
    for ax in axes:
        empty_axis(ax)
axes[0].set(
    title="Overlap bbox maksimum per gambar",
    xlabel="IoU maksimum (0–1)",
    ylabel="Jumlah gambar (file)",
)
axes[1].set(
    title="Jarak pusat terdekat",
    xlabel="Jarak Euclidean ternormalisasi (0–√2)",
    ylabel="Jumlah bbox (instance)",
)
finish_figure(
    fig,
    "eda_10b_overlap_and_distance",
    f"Seluruh pasangan hanya pada gambar dengan 2–{PAIRWISE_MAX_OBJECTS} bbox; kotak identik dapat menghasilkan IoU=1.",
)
interpret(
    "GEOMETRI",
    f"{int(pair_df['status'].eq('diperiksa').sum()):,} gambar diperiksa; "
    f"{int(pair_df['status'].eq('di atas batas biaya').sum()):,} gambar dilewati karena batas biaya. "
    f"Dihitung {int(pair_df['pair_count'].sum()):,} pasangan dan {len(nearest_df):,} jarak terdekat. "
    "Statistik tidak mewakili gambar yang dilewati.",
)


## 11. Hubungan dan komposisi antarkelas

### 11.1 Ko-kemunculan kelas — EDA-14

**Tujuan:** Mendeskripsikan kelas yang muncul bersama pada gambar yang sama.

**Metode:** Matriks biner kehadiran kelas per gambar berlabel lengkap. Cij adalah jumlah gambar yang memuat kelas i dan j; diagonal adalah jumlah gambar yang memuat kelas i. Jaccard=Cij/(Cii+Cjj−Cij), NaN jika union nol. Background berada dalam denominator gambar namun tidak menambah Cij.

**Caption:** Heatmap jumlah gambar ko-kemunculan dan Jaccard; tabel gabungan serta per split.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Ko-kemunculan membantu menjelaskan komposisi adegan tanpa dianggap hubungan kausal atau confusion model.

In [ ]:
def class_presence(
    boxes: pd.DataFrame, image_ids: list[str], class_ids: list[int]
) -> pd.DataFrame:
    """Matriks biner gambar×kelas; background tetap ada sebagai baris nol."""
    if boxes.empty:
        return pd.DataFrame(0, index=image_ids, columns=class_ids, dtype=np.int64)
    return (
        pd.crosstab(boxes["image_id"], boxes["class_id"]).reindex(
            index=image_ids, columns=class_ids, fill_value=0
        )
        > 0
    ).astype(np.int64)


presence_df = class_presence(boxes_df, count_images_df["image_id"].tolist(), CLASS_IDS)
cooccurrence = presence_df.to_numpy().T @ presence_df.to_numpy()
diagonal = np.diag(cooccurrence)
union = diagonal[:, None] + diagonal[None, :] - cooccurrence
jaccard = np.divide(
    cooccurrence,
    union,
    out=np.full(cooccurrence.shape, np.nan, dtype=float),
    where=union > 0,
)
register_table(
    "cooccurrence",
    pd.DataFrame(cooccurrence, index=CLASS_NAMES, columns=CLASS_NAMES)
    .rename_axis("kelas")
    .reset_index(),
)
register_table(
    "cooccurrence_jaccard",
    pd.DataFrame(jaccard, index=CLASS_NAMES, columns=CLASS_NAMES)
    .rename_axis("kelas")
    .reset_index(),
)
cooccurrence_rows = []
for split in SPLITS:
    ids = count_images_df.loc[count_images_df["split"].eq(split), "image_id"]
    values = presence_df.reindex(ids).to_numpy()
    matrix = values.T @ values
    cooccurrence_rows.extend(
        {
            "split": split,
            "class_id_i": i,
            "class_id_j": j,
            "kelas_i": CLASS_NAMES[i],
            "kelas_j": CLASS_NAMES[j],
            "gambar_bersama": int(matrix[i, j]),
            "denominator_gambar": len(ids),
        }
        for i in CLASS_IDS
        for j in CLASS_IDS
    )
register_table("cooccurrence_by_split", pd.DataFrame(cooccurrence_rows), show=False)
fig, axes = plt.subplots(
    1, 2, figsize=(max(14, 2 * len(CLASS_NAMES)), max(5, len(CLASS_NAMES) * 0.65))
)
for ax, matrix, title, unit in zip(
    axes,
    [cooccurrence, jaccard],
    ["Jumlah gambar ko-kemunculan", "Kemiripan kehadiran kelas"],
    ["Jumlah gambar (file)", "Jaccard (0–1)"],
):
    plotted = ax.imshow(
        matrix, cmap="Blues", vmin=0, vmax=1 if unit.startswith("Jaccard") else None
    )
    ax.set_xticks(CLASS_IDS, CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(CLASS_IDS, CLASS_NAMES)
    ax.set_title(title)
    fig.colorbar(plotted, ax=ax, label=unit, shrink=0.8)
    if len(CLASS_NAMES) <= 12:
        for i in CLASS_IDS:
            for j in CLASS_IDS:
                value = matrix[i, j]
                text = (
                    "NA"
                    if np.isnan(value)
                    else f"{value:.2f}"
                    if unit.startswith("Jaccard")
                    else str(int(value))
                )
                finite_values = matrix[np.isfinite(matrix)]
                maximum = finite_values.max() if len(finite_values) else 1
                ax.text(
                    j,
                    i,
                    text,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white"
                    if np.isfinite(value) and value > maximum * 0.6
                    else "black",
                )
finish_figure(
    fig,
    "eda_14_class_cooccurrence",
    "Diagonal memuat kehadiran kelas pada gambar berlabel lengkap; NA berarti union nol.",
)
interpret(
    "EDA-14",
    f"Matriks memakai {len(presence_df):,} gambar berlabel lengkap. "
    f"{int((presence_df.sum(axis=1) >= 2).sum()):,} gambar memuat sekurangnya dua kelas. "
    "Matriks jumlah gambar lebih dipengaruhi kelas sering muncul daripada Jaccard.",
)


## 12. Visualisasi sampel dan ground truth

### 12.1 Sampel representatif dengan anotasi asli — EDA-15

**Tujuan:** Memberikan konteks visual dengan pemilihan contoh yang dapat diulang.

**Metode:** Contoh dipilih dari gambar terverifikasi berlabel lengkap dan orientasi EXIF standar: count rendah/tinggi, kelas paling jarang yang hadir, brightness minimum/maksimum yang diukur, satu gambar tiap split bila kapasitas cukup, lalu sampel acak seed tetap. Contoh ekstrem bersifat purposive dan tidak dianggap sampel representatif statistik. Kotak memakai normalized xywh→xyxy pixel asli tanpa clipping.

**Caption:** Grid ground truth mencantumkan partisi, count, alasan pemilihan, dan nama kelas dengan warna konsisten.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Contoh mendukung dokumentasi dataset; anotasi yang terlihat berasal dari label, bukan prediksi model.

In [ ]:
def select_example_images() -> pd.DataFrame:
    """Pilih ekstrem dan contoh acak deterministik tanpa duplikasi image_id."""
    pool = images_df.loc[
        images_df["readable"]
        & images_df["annotation_complete"]
        & images_df["exif_orientation"].isin([1])
    ].copy()
    selections = {}

    def add(index: Any, reason: str) -> None:
        if index is not None and len(selections) < SAMPLE_GRID_SIZE:
            selections.setdefault(index, []).append(reason)

    if pool.empty:
        return pd.DataFrame(columns=[*IMAGE_COLUMNS, "reason"])
    add(pool["n_objects"].idxmin(), "jumlah objek paling sedikit")
    add(pool["n_objects"].idxmax(), "jumlah objek paling banyak")
    class_counts = (
        boxes_df.loc[boxes_df["image_id"].isin(pool["image_id"])]
        .groupby("class_id")
        .size()
    )
    if len(class_counts):
        rare_id = int(class_counts.idxmin())
        rare_images = pool.loc[
            pool["image_id"].isin(
                boxes_df.loc[boxes_df["class_id"].eq(rare_id), "image_id"]
            )
        ]
        if len(rare_images):
            add(rare_images.index[0], f"kelas jarang: {CLASS_NAMES[rare_id]}")
    measured = pool.loc[pool["brightness"].notna()]
    if len(measured):
        add(
            measured["brightness"].idxmin(),
            "brightness terendah pada cakupan pengukuran",
        )
        add(
            measured["brightness"].idxmax(),
            "brightness tertinggi pada cakupan pengukuran",
        )
    for split in SPLITS:
        part = pool.loc[pool["split"].eq(split)]
        if len(part):
            add(part.index[0], f"contoh partisi {SPLIT_LABELS[split]}")
    for index in np.random.default_rng(SEED).permutation(pool.index):
        add(index, "sampel acak deterministik")
    selected = pool.loc[list(selections)].copy()
    selected["reason"] = ["; ".join(selections[index]) for index in selected.index]
    return selected


example_df = select_example_images()
register_table(
    "representative_samples",
    example_df[["image_id", "split", "image_path", "n_objects", "reason"]],
)
columns = 2
rows = max(1, math.ceil(len(example_df) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(16, 5 * rows), squeeze=False)
for ax, record in zip(axes.flat, example_df.itertuples()):
    path = Path(record.image_path)
    try:
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            width, height = rgb.size
            ax.imshow(rgb)
        for box in boxes_df.loc[boxes_df["image_id"].eq(record.image_id)].itertuples():
            left, top, right, bottom = xywh_to_pixels(
                box.x_center,
                box.y_center,
                box.width_norm,
                box.height_norm,
                width,
                height,
            )
            color = CLASS_COLORS[box.class_id]
            ax.add_patch(
                Rectangle(
                    (left, top),
                    right - left,
                    bottom - top,
                    edgecolor=color,
                    facecolor="none",
                    linewidth=1.4,
                )
            )
            ax.text(
                left,
                top,
                CLASS_NAMES[box.class_id],
                fontsize=8,
                va="bottom",
                color="white",
                bbox={"facecolor": color, "alpha": 0.85, "edgecolor": "none", "pad": 1},
            )
        title = f"{SPLIT_LABELS[record.split]} | {int(record.n_objects)} objek | {path.name}"
        ax.set_title(
            textwrap.fill(title, width=68)
            + "\n"
            + textwrap.fill(record.reason, width=68),
            fontsize=9,
        )
    except (OSError, ValueError, Image.DecompressionBombError) as error:
        add_issue(record.split, path, "decode_sampel_gagal", "CRITICAL", str(error))
        empty_axis(ax, f"Contoh tidak dapat didekode: {error}")
    ax.axis("off")
for ax in list(axes.flat)[len(example_df) :]:
    ax.axis("off")
if example_df.empty:
    empty_axis(
        axes.flat[0], "Tidak ada gambar yang memenuhi syarat visualisasi ground truth"
    )
finish_figure(
    fig,
    "eda_15_ground_truth_samples",
    f"Ground truth asli, pemilihan seed {SEED}; alasan pemilihan diekspor dan ekstrem tidak dianggap sampel acak populasi.",
)
interpret(
    "EDA-15",
    f"Ditampilkan {len(example_df):,} contoh. Gambar dengan label bermasalah "
    "atau orientasi EXIF nonstandar tidak digunakan sebagai ilustrasi ground truth valid.",
)


## 13. Audit potensi data leakage

### 13.1 Identitas path, SHA-256, dan konsistensi label antarsplit

**Tujuan:** Menemukan keanggotaan sumber yang berulang atau citra identik lintas partisi.

**Metode:** Path teresolusi yang sama lintas split diperiksa pada semua mode. SHA-256 byte citra berjalan jika diaktifkan; dilakukan sekali per path fisik unik secara streaming. Hash label selalu dihitung. Label berbeda untuk citra dengan hash identik memerlukan inspeksi karena kesamaan byte label lebih ketat daripada kesamaan semantik. Kegagalan hash dan cakupan dicatat.

**Caption:** Tabel kelompok path/hash duplikat mencantumkan anggota, split, dan konsistensi byte label; status audit memisahkan pemeriksaan yang dijalankan dan dilewati.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** SHA-256 membuktikan kesamaan byte dan tidak menangkap seluruh bentuk kebocoran, seperti frame berdekatan atau sumber rekaman sama.

In [ ]:
leak_started = time.perf_counter()
duplicate_rows = []
label_digest_map = (
    labels_df.drop_duplicates("label_path").set_index("label_path")["sha256"].to_dict()
)
shared_paths = images_df.groupby("image_path", sort=False)
for path, part in shared_paths:
    if part["split"].nunique() > 1:
        add_issue(
            "lintas split",
            path,
            "path_gambar_sama_antarsplit",
            "CRITICAL",
            "Sumber gambar sama terdaftar dalam beberapa split.",
            occurrences=len(part) - 1,
        )
        for image in part.itertuples():
            duplicate_rows.append(
                {
                    "method": "path",
                    "group": path,
                    "image_id": image.image_id,
                    "split": image.split,
                    "image_path": path,
                    "label_sha256": label_digest_map.get(image.label_path),
                    "cross_split": True,
                    "label_byte_consistency": "path label sama",
                }
            )
hash_failures = 0
if HASH_ENABLED:
    digests = {}
    for path in sorted(images_df["image_path"].unique()):
        try:
            digests[path] = sha256_file(Path(path))
        except OSError as error:
            hash_failures += 1
            add_issue("semua", path, "hash_gambar_gagal", "WARNING", str(error))
    images_df["sha256"] = images_df["image_path"].map(digests)
    for digest, part in images_df.dropna(subset=["sha256"]).groupby(
        "sha256", sort=False
    ):
        if len(part) < 2:
            continue
        cross_split = part["split"].nunique() > 1
        label_hashes = [label_digest_map.get(path) for path in part["label_path"]]
        known_hashes = [value for value in label_hashes if value is not None]
        consistency = (
            "tidak dapat dibandingkan"
            if len(known_hashes) != len(label_hashes)
            else "byte label sama"
            if len(set(known_hashes)) == 1
            else "byte label berbeda; inspeksi diperlukan"
        )
        issue = (
            "sha256_identik_antarsplit" if cross_split else "sha256_identik_dalam_split"
        )
        add_issue(
            "lintas split" if cross_split else part.iloc[0]["split"],
            part.iloc[0]["image_path"],
            issue,
            "CRITICAL" if cross_split else "WARNING",
            f"SHA-256={digest}; {consistency}",
            occurrences=len(part) - 1,
        )
        if consistency.startswith("byte label berbeda"):
            add_issue(
                "lintas split" if cross_split else part.iloc[0]["split"],
                part.iloc[0]["image_path"],
                "label_berbeda_pada_gambar_identik",
                "WARNING",
                consistency,
            )
        for image in part.itertuples():
            duplicate_rows.append(
                {
                    "method": "sha256",
                    "group": digest,
                    "image_id": image.image_id,
                    "split": image.split,
                    "image_path": image.image_path,
                    "label_sha256": label_digest_map.get(image.label_path),
                    "cross_split": cross_split,
                    "label_byte_consistency": consistency,
                }
            )
duplicates_df = pd.DataFrame(
    duplicate_rows,
    columns=[
        "method",
        "group",
        "image_id",
        "split",
        "image_path",
        "label_sha256",
        "cross_split",
        "label_byte_consistency",
    ],
)
register_table("exact_duplicate_groups", duplicates_df)
hash_coverage = pd.DataFrame(
    [
        {
            "method": "path lintas split",
            "status": "dijalankan",
            "referensi": len(images_df),
        },
        {
            "method": "SHA-256 gambar",
            "status": "dijalankan" if HASH_ENABLED else "tidak dijalankan",
            "referensi": int(images_df["sha256"].notna().sum()),
            "gagal_path_unik": hash_failures,
        },
    ]
)
register_table("leakage_audit_coverage", hash_coverage)
TIMINGS["audit_hash"] = time.perf_counter() - leak_started
interpret(
    "LEAKAGE",
    f"Kelompok duplikat lintas split yang dilaporkan: "
    f"{duplicates_df.loc[duplicates_df['cross_split'], ['method', 'group']].drop_duplicates().shape[0]:,} "
    "(kelompok path dan hash dapat merujuk citra yang sama). "
    + (
        f"SHA-256 tersedia untuk {images_df['sha256'].notna().sum():,} referensi gambar."
        if HASH_ENABLED
        else "**SHA-256 gambar tidak dijalankan pada konfigurasi ini; audit kesamaan byte belum lengkap.**"
    )
    + " Tidak ada temuan tidak berarti seluruh bentuk leakage telah disingkirkan.",
)


### 13.2 Kandidat kemiripan visual opsional

**Tujuan:** Mengidentifikasi kandidat citra mirip dengan biaya komputasi terkontrol.

**Metode:** dHash 64 bit dihitung dari grayscale resize 9×8; bit menyatakan perbandingan pixel bertetangga. Kandidat hanya lintas split dan jarak Hamming≤ambang. Sampel path unik terverifikasi maksimal batas total memakai seed tetap; tidak ada koreksi untuk prevalensi populasi. Batas pasangan hasil mencegah output/RAM tak terbatas; penghentian dini dinyatakan.

**Caption:** Daftar kandidat memuat dua path, split, hash, dan jarak Hamming; coverage memuat dipilih/berhasil dan truncation.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Hash perseptual hanya menghasilkan kandidat inspeksi, termasuk false positive; bukan bukti duplikasi semantik atau leakage pasti.

In [ ]:
def image_dhash(path: Path) -> int:
    """dHash horizontal 64 bit pada pixel grayscale yang di-resize menjadi 9×8."""
    with Image.open(path) as image:
        array = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.LANCZOS))
    bits = (array[:, 1:] > array[:, :-1]).ravel()
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


near_started = time.perf_counter()
near_rows, perceptual_records = [], []
near_selected, near_comparisons, near_truncated = 0, 0, False
if RUN_NEAR_DUPLICATES:
    candidates = (
        images_df.loc[images_df["readable"]]
        .drop_duplicates("image_path")
        .sort_values("image_path")
    )
    count = min(len(candidates), NEAR_DUPLICATE_MAX_IMAGES)
    indices = np.sort(
        np.random.default_rng(SEED).choice(len(candidates), count, replace=False)
    )
    selected = candidates.iloc[indices]
    near_selected = len(selected)
    for record in selected.itertuples():
        try:
            digest = image_dhash(Path(record.image_path))
            splits_for_path = set(
                images_df.loc[images_df["image_path"].eq(record.image_path), "split"]
            )
            perceptual_records.append((record.image_path, splits_for_path, digest))
            images_df.loc[images_df["image_path"].eq(record.image_path), "dhash"] = (
                f"{digest:016x}"
            )
        except (OSError, ValueError, Image.DecompressionBombError) as error:
            add_issue(
                record.split, record.image_path, "dhash_gagal", "WARNING", str(error)
            )
    for first_index, (first_path, first_splits, first_hash) in enumerate(
        perceptual_records
    ):
        if near_truncated:
            break
        for second_path, second_splits, second_hash in perceptual_records[
            first_index + 1 :
        ]:
            if len(first_splits | second_splits) < 2:
                continue
            near_comparisons += 1
            distance = (first_hash ^ second_hash).bit_count()
            if distance <= DHASH_MAX_DISTANCE:
                near_rows.append(
                    {
                        "path_a": first_path,
                        "path_b": second_path,
                        "split_a": ",".join(sorted(first_splits)),
                        "split_b": ",".join(sorted(second_splits)),
                        "dhash_a": f"{first_hash:016x}",
                        "dhash_b": f"{second_hash:016x}",
                        "hamming_distance": distance,
                    }
                )
                if len(near_rows) >= NEAR_DUPLICATE_MAX_PAIRS:
                    near_truncated = True
                    break
near_df = pd.DataFrame(
    near_rows,
    columns=[
        "path_a",
        "path_b",
        "split_a",
        "split_b",
        "dhash_a",
        "dhash_b",
        "hamming_distance",
    ],
)
register_table("near_duplicate_candidates", near_df)
register_table(
    "near_duplicate_coverage",
    pd.DataFrame(
        [
            {
                "enabled": RUN_NEAR_DUPLICATES,
                "selected_unique_paths": near_selected,
                "successful_hashes": len(perceptual_records),
                "cross_split_comparisons": near_comparisons,
                "candidate_pairs": len(near_df),
                "truncated": near_truncated,
                "max_images": NEAR_DUPLICATE_MAX_IMAGES,
                "max_pairs": NEAR_DUPLICATE_MAX_PAIRS,
                "hamming_threshold": DHASH_MAX_DISTANCE,
            }
        ]
    ),
)
if len(near_df):
    add_issue(
        "lintas split",
        "lihat near_duplicate_candidates.csv",
        "kandidat_kemiripan_visual",
        "INFO",
        "Kandidat dHash perlu inspeksi; bukan bukti duplikasi pasti.",
        occurrences=len(near_df),
    )
TIMINGS["audit_perseptual"] = time.perf_counter() - near_started
interpret(
    "NEAR-DUPLICATE",
    (
        f"{len(near_df):,} kandidat dari {len(perceptual_records):,} path yang berhasil "
        f"di-hash; {near_comparisons:,} pasangan diperiksa; truncation={near_truncated}. "
        if RUN_NEAR_DUPLICATES
        else "Pemeriksaan dHash tidak diaktifkan. "
    )
    + "Kandidat memerlukan inspeksi manual; cakupan sampel tidak dianggap cakupan penuh.",
)


## 14. Ringkasan temuan EDA

### 14.1 Ringkasan kualitas anotasi — EDA-16

**Tujuan:** Menyatukan hasil audit sebelum digunakan dalam penulisan ilmiah.

**Metode:** Temuan dikelompokkan menurut jenis masalah dan severity, menjumlahkan occurrences. Satu baris/file dapat memiliki beberapa temuan; jumlah kejadian bukan jumlah file unik. Status label dilaporkan per referensi gambar. Background valid dibedakan dari label hilang dan citra rusak; kandidat kemiripan tidak dihitung sebagai kegagalan pasti.

**Caption:** Grafik status label per split, tabel status integritas, serta laporan masalah per file.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Temuan kritis membutuhkan pemeriksaan sumber, tanpa mengubah data atau menyatakan semua eksperimen otomatis tidak valid.

In [ ]:
issues_df = pd.DataFrame(ISSUE_ROWS, columns=ISSUE_COLUMNS)
audit_summary = (
    issues_df.groupby(["severity", "issue"], sort=False)["occurrences"]
    .sum()
    .reset_index()
)
register_table("annotation_issues", issues_df, show=False)
register_table("annotation_quality_summary", audit_summary)
status_order = [
    "label valid",
    "label kosong",
    "label hilang",
    "label tidak valid",
    "label tidak terbaca",
]
status_table = pd.crosstab(images_df["split"], images_df["label_status"]).reindex(
    index=SPLITS, columns=status_order, fill_value=0
)
register_table("annotation_status", status_table.rename_axis("split").reset_index())
integrity_table = (
    images_df.groupby(["split", "readable", "integrity_level"])
    .size()
    .reset_index(name="gambar")
)
register_table("image_integrity", integrity_table)
fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(SPLITS))
status_colors = ["#0072B2", "#009E73", "#E69F00", "#D55E00", "#CC79A7"]
for status, color in zip(status_order, status_colors):
    counts = status_table[status].to_numpy()
    ax.bar(
        [SPLIT_LABELS[split] for split in SPLITS],
        counts,
        bottom=bottom,
        label=status,
        color=color,
    )
    bottom += counts
ax.set(title="Status label per partisi", ylabel="Jumlah referensi gambar (file)")
ax.legend(title="Status label", bbox_to_anchor=(1.02, 1), loc="upper left")
finish_figure(
    fig,
    "eda_16_annotation_quality",
    "Label kosong menjadi background terverifikasi hanya bila citra terbaca dan pasangan unik; integritas citra dilaporkan terpisah.",
)
critical_issues = issues_df.loc[issues_df["severity"].eq("CRITICAL")]
interpret(
    "EDA-16",
    f"Audit memuat {len(issues_df):,} baris temuan, "
    f"{int(issues_df['occurrences'].sum()):,} kejadian, dan "
    f"{issues_df['file'].nunique():,} file/lokasi temuan unik. "
    f"Kejadian kritis: {int(critical_issues['occurrences'].sum()):,}. "
    "Tidak ada perbaikan, clipping, deduplikasi, atau perubahan split otomatis.",
)
if len(critical_issues):
    display(
        Markdown(
            "**Temuan kritis memerlukan inspeksi sebelum angka dipakai sebagai deskripsi dataset final.**"
        )
    )
    display(critical_issues.head(20))


### 14.2 Tabel hasil utama dan batas interpretasi

**Tujuan:** Menyediakan angka yang bisa ditelusuri untuk penulisan Bab III/V/VI.

**Metode:** Ringkasan dihitung dari tabel mentah gambar, bbox valid, audit, dan coverage. Setiap indikator menyebut denominator serta sumber tabel. Narasi otomatis menggunakan nilai runtime; nilai belum tersedia tetap NaN/tidak tersedia.

**Caption:** Tabel indikator utama dan narasi yang seluruhnya berasal dari eksekusi dataset sumber.

**Interpretasi:** Narasi berikut dihasilkan dari tabel pada saat eksekusi.

**Implikasi:** Interpretasi dibatasi pada dataset dan cakupan pengukuran; tidak ada metrik performa atau peringkat metode KD.

In [ ]:
summary_rows = [
    {
        "indikator": "Jumlah gambar terinventarisasi",
        "nilai": len(images_df),
        "satuan": "referensi gambar unik per split",
        "denominator": "seluruh split tersedia",
        "sumber": "image_inventory",
    },
    {
        "indikator": "Jumlah bbox valid",
        "nilai": len(boxes_df),
        "satuan": "baris anotasi",
        "denominator": "pasangan unik; duplikat valid termasuk",
        "sumber": "bounding_boxes",
    },
    {
        "indikator": "Gambar berlabel lengkap",
        "nilai": len(count_images_df),
        "satuan": "gambar",
        "denominator": str(len(images_df)),
        "sumber": "image_inventory",
    },
    {
        "indikator": "Median objek per gambar",
        "nilai": count_images_df["n_objects"].median(),
        "satuan": "instance/gambar",
        "denominator": str(len(count_images_df)),
        "sumber": "object_count_statistics",
    },
    {
        "indikator": "Median luas relatif bbox",
        "nilai": boxes_df["area_relative"].median(),
        "satuan": "luas bbox/luas citra",
        "denominator": str(len(boxes_df)),
        "sumber": "bounding_boxes",
    },
    {
        "indikator": "Median brightness terukur",
        "nilai": brightness_df["brightness"].median(),
        "satuan": "grayscale 0–255",
        "denominator": str(len(brightness_df)),
        "sumber": "image_inventory",
    },
    {
        "indikator": "Kejadian audit kritis",
        "nilai": int(critical_issues["occurrences"].sum()),
        "satuan": "kejadian",
        "denominator": "satu file dapat memiliki beberapa kejadian",
        "sumber": "annotation_issues",
    },
    {
        "indikator": "Gambar dengan SHA-256",
        "nilai": int(images_df["sha256"].notna().sum()),
        "satuan": "referensi gambar",
        "denominator": str(len(images_df)),
        "sumber": "image_inventory",
    },
]
main_summary = register_table("main_findings", pd.DataFrame(summary_rows))
findings_table = register_table("interpretations", pd.DataFrame(FINDINGS))
display(
    Markdown(
        "**Batas pemakaian:** statistik kelas dapat memuat baris valid dari label sebagian "
        "bermasalah; distribusi count dan ko-kemunculan memakai label lengkap. "
        "Citra lolos verify pada standard belum semuanya didekode. Hasil brightness "
        "standard adalah statistik sampel per split. Tidak ada hasil yang menyatakan metode KD tertentu lebih unggul."
    )
)


## 15. Ekspor hasil dan provenance

### 15.1 Tabel mentah, laporan, fingerprint, dan manifest

**Tujuan:** Membuat hasil dapat ditelusuri dan dipakai untuk dokumentasi skripsi.

**Metode:** Seluruh tabel terdaftar serta inventaris gambar, inventaris label, baris
label mentah, dan bbox valid diekspor CSV UTF-8. Laporan Markdown/JSON memuat angka
aktual dan interpretasi runtime. Provenance mencatat YAML asli/hash, metadata versi
yang tersedia, library, konfigurasi, parameter sampling, resolusi path, dan coverage.
Fingerprint inventaris merangkum path relatif, ukuran file, dan hash label; tanpa
SHA-256 gambar penuh fingerprint tersebut tidak membuktikan identitas konten citra.
Manifest memberi SHA-256 setiap artefak hasil yang dibuat run ini, tanpa memasukkan
file lama yang mungkin berada dalam direktori output.

**Caption:** Struktur ekspor `figures/`, `tables/`, `reports/`, dan `manifest.json`
di `/kaggle/working/eda_poltrada_v3/`.

**Interpretasi dan implikasi:** Hanya hasil yang diekspor pada run ini masuk manifest.
Notebook tidak membutuhkan file Python proyek atau jaringan; direktori Input tetap read-only.

In [ ]:
def json_safe(value: Any) -> Any:
    """Konversi tipe numerik dan nilai nonfinite agar JSON standar tidak mengandung NaN."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, value: Any) -> None:
    """Simpan UTF-8 JSON valid; nilai tak hingga/tidak tersedia menjadi null."""
    path.write_text(
        json.dumps(json_safe(value), indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )


def markdown_table(frame: pd.DataFrame) -> str:
    """Render tabel Markdown tanpa dependency tabulate dan escape pemisah kolom."""

    def format_value(value: Any) -> str:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return "Tidak tersedia"
        return str(value).replace("|", "\\|").replace("\n", " ")

    header = "| " + " | ".join(map(str, frame.columns)) + " |"
    divider = "| " + " | ".join("---" for _ in frame.columns) + " |"
    rows = [
        "| " + " | ".join(format_value(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    ]
    return "\n".join([header, divider, *rows])


export_started = time.perf_counter()
table_dir, report_dir = OUTPUT_DIR / "tables", OUTPUT_DIR / "reports"
table_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)
register_table("image_inventory", images_df, show=False)
register_table("label_inventory", labels_df, show=False)
register_table("raw_label_rows", raw_labels_df, show=False)
register_table("bounding_boxes", boxes_df, show=False)
exported_paths = []
for name, table in TABLES.items():
    path = table_dir / f"{name}.csv"
    table.to_csv(path, index=False, encoding="utf-8-sig")
    exported_paths.append(path)
audit_report_path = report_dir / "audit_report.csv"
issues_df.to_csv(audit_report_path, index=False, encoding="utf-8-sig")
exported_paths.append(audit_report_path)


def relative_source(path_string: str) -> str:
    path = Path(path_string)
    return (
        path.relative_to(DATA_ROOT).as_posix()
        if path.is_relative_to(DATA_ROOT)
        else path.as_posix()
    )


inventory_digest = hashlib.sha256()
for table in (images_df, labels_df):
    path_column = "image_path" if "image_path" in table else "label_path"
    for row in table.sort_values(["split", path_column]).to_dict(orient="records"):
        record = {
            "split": row["split"],
            "path": relative_source(row[path_column]),
            "file_bytes": row.get("file_bytes"),
            "sha256": row.get("sha256"),
        }
        inventory_digest.update(
            json.dumps(json_safe(record), sort_keys=True).encode("utf-8")
        )
        inventory_digest.update(b"\n")

configuration = {
    "mode": EDA_MODE,
    "seed": SEED,
    "brightness_sample_per_split": BRIGHTNESS_SAMPLE_PER_SPLIT,
    "brightness_thumbnail_side": BRIGHTNESS_THUMBNAIL_SIDE,
    "image_sha256_enabled": HASH_ENABLED,
    "near_duplicate_enabled": RUN_NEAR_DUPLICATES,
    "near_max_images": NEAR_DUPLICATE_MAX_IMAGES,
    "near_max_pairs": NEAR_DUPLICATE_MAX_PAIRS,
    "dhash_max_distance": DHASH_MAX_DISTANCE,
    "pairwise_max_objects": PAIRWISE_MAX_OBJECTS,
    "relative_area_thresholds": RELATIVE_AREA_THRESHOLDS,
    "spatial_bins": SPATIAL_BINS,
    "sample_grid_size": SAMPLE_GRID_SIZE,
    "coordinate_tolerance": COORDINATE_TOLERANCE,
    "dpi": FIGURE_DPI,
    "save_svg": SAVE_SVG,
}
provenance = {
    "dataset_name": DATASET_NAME,
    "yaml_source": str(DATA_YAML),
    "yaml_sha256": sha256_file(DATA_YAML),
    "yaml_content": DATA_CONFIG,
    "dataset_version_metadata": {
        key: DATA_CONFIG[key]
        for key in ("version", "roboflow", "license", "url")
        if key in DATA_CONFIG
    },
    "dataset_version_note": "Nama Poltrada V3 berasal dari konfigurasi sumber; versi formal hanya bila metadata YAML tersedia.",
    "resolved_root": str(DATA_ROOT),
    "root_override": DATA_ROOT_OVERRIDE,
    "class_names": CLASS_NAMES,
    "nc": len(CLASS_NAMES),
    "splits": SPLITS,
    "test_available": "test" in SPLITS,
    "source_resolution": SOURCE_TABLE.to_dict(orient="records"),
    "run_started_utc": RUN_STARTED,
    "python": platform.python_version(),
    "libraries": VERSIONS,
    "configuration": configuration,
    "file_counts": {
        "image_references": len(images_df),
        "unique_image_paths": images_df["image_path"].nunique(),
        "label_path_references": len(labels_df),
        "existing_label_paths": labels_df.loc[
            labels_df["sha256"].notna(), "label_path"
        ].nunique(),
    },
    "inventory_fingerprint_sha256": inventory_digest.hexdigest(),
    "fingerprint_scope": "path, ukuran file citra, hash label; hash citra hanya jika audit SHA-256 aktif",
    "coverage": {
        "brightness": TABLES["brightness_coverage"].to_dict(orient="records"),
        "pairwise": TABLES["pairwise_coverage"].to_dict(orient="records"),
        "leakage": hash_coverage.to_dict(orient="records"),
        "near_duplicate": TABLES["near_duplicate_coverage"].to_dict(orient="records"),
    },
}
summary = {
    "dataset": DATASET_NAME,
    "split_summary": split_summary.to_dict(orient="records"),
    "class_distribution": class_distribution.to_dict(orient="records"),
    "imbalance": imbalance_table.to_dict(orient="records"),
    "main_findings": main_summary.to_dict(orient="records"),
    "interpretations": FINDINGS,
    "critical_occurrences": int(critical_issues["occurrences"].sum()),
    "audit_summary": audit_summary.to_dict(orient="records"),
    "limitations": [
        "EDA tidak mengukur performa atau keunggulan metode KD.",
        "Statistik bbox hanya baris valid dari pasangan unik; duplikat valid tidak dibuang.",
        "Label hilang bukan bukti background; count/ko-kemunculan memakai label lengkap.",
        "Brightness standard adalah sampel deterministik dengan estimator thumbnail.",
        "Verify standard tidak menjamin semua pixel dapat didekode.",
        "SHA-256 hanya membuktikan kesamaan byte pada cakupan yang diperiksa.",
        "dHash menghasilkan kandidat inspeksi dengan kemungkinan false positive.",
        "Geometri pasangan melewati gambar di atas batas objek.",
        "Kategori ukuran relatif bukan kategori COCO atau ukuran setelah letterbox.",
    ],
}
summary_lines = [
    f"# Ringkasan EDA {DATASET_NAME}",
    "",
    f"Sumber YAML: `{DATA_YAML}`",
    "",
    "## Pembagian dataset",
    "",
    markdown_table(split_summary),
    "",
    "## Distribusi kelas keseluruhan",
    "",
    markdown_table(class_total),
    "",
    "## Indikator utama",
    "",
    markdown_table(main_summary),
    "",
    "## Audit kualitas",
    "",
    markdown_table(audit_summary),
    "",
    "## Interpretasi berdasarkan runtime",
    "",
]
summary_lines.extend(
    f"- **{finding['kode']}:** {finding['interpretasi']}" for finding in FINDINGS
)
summary_lines.extend(["", "## Batas interpretasi", ""])
summary_lines.extend(f"- {limitation}" for limitation in summary["limitations"])
TIMINGS["ekspor_tabel"] = time.perf_counter() - export_started
elapsed_to_report = time.perf_counter() - START_TIME
summary["runtime_seconds_to_report"] = elapsed_to_report
provenance["runtime_seconds_to_report"] = elapsed_to_report
provenance["timings_seconds"] = TIMINGS
summary_lines.extend(
    [
        "",
        (
            f"Durasi hingga penyusunan laporan: {elapsed_to_report:.2f} detik. "
            "Durasi akhir termasuk hashing artefak terdapat pada manifest."
        ),
    ]
)
summary_md_path = report_dir / "eda_summary.md"
summary_md_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
exported_paths.append(summary_md_path)
for name, value in [("eda_summary", summary), ("dataset_provenance", provenance)]:
    path = report_dir / f"{name}.json"
    write_json(path, value)
    exported_paths.append(path)
for record in FIGURE_RECORDS:
    exported_paths.extend(
        OUTPUT_DIR / "figures" / f"{record['name']}.{extension}"
        for extension in record["formats"]
    )
artifact_rows = [
    {
        "path": path.relative_to(OUTPUT_DIR).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in sorted(set(exported_paths))
]
manifest = {
    "dataset": DATASET_NAME,
    "run_started_utc": RUN_STARTED,
    "runtime_seconds": time.perf_counter() - START_TIME,
    "configuration": configuration,
    "yaml_sha256": provenance["yaml_sha256"],
    "inventory_fingerprint_sha256": inventory_digest.hexdigest(),
    "figures": FIGURE_RECORDS,
    "artifacts": artifact_rows,
    "manifest_scope": "Hanya artefak yang dibuat run ini; manifest tidak meng-hash dirinya sendiri.",
}
write_json(OUTPUT_DIR / "manifest.json", manifest)
display(
    Markdown(
        f"**Ekspor selesai:** `{OUTPUT_DIR}`. {len(FIGURE_RECORDS)} kelompok grafik, "
        f"{len(TABLES)} tabel, {len(artifact_rows)} artefak terdaftar. "
        f"Durasi hingga manifest {manifest['runtime_seconds']:.2f} detik."
    )
)


## 16. Kesimpulan dan implikasi bagi penelitian KD

### 16.1 Sintesis deskriptif berdasarkan dataset sumber

**Tujuan:** Menghubungkan karakteristik data dengan kebutuhan dokumentasi penelitian.

**Metode:** Sintesis memakai angka ringkasan yang telah dihitung. Ketidakseimbangan
kelas, jumlah objek, skala relatif, brightness, pola spasial, dan audit dilaporkan
sebagai konteks pengamatan. Tidak ada hipotesis performa yang dinyatakan terbukti.

**Caption:** Kesimpulan runtime dan lokasi laporan untuk penulisan Bab III, V, dan VI.

**Implikasi:** Jumlah gambar/instance serta pembagian partisi mendukung Bab III;
provenance, audit, dan contoh ground truth mendukung Bab V; distribusi kelas/skala
serta keterbatasan mendukung pembahasan hasil Bab VI. Penilaian tiga metode KD tetap
memerlukan evaluasi model sesuai protokol eksperimen yang telah ditetapkan.

Gunakan tabel hasil, bukan angka dari acuan akademik atau fixture pengujian.
Temuan data harus diperiksa dan didokumentasikan sebelum menjadi keputusan penelitian.

In [ ]:
display(
    Markdown(
        f"Dataset sumber memuat **{len(images_df):,} referensi gambar** pada "
        f"**{len(SPLITS)} partisi**, dengan **{len(CLASS_NAMES)} kelas dari YAML** dan "
        f"**{len(boxes_df):,} baris bbox valid** dari pasangan unik. "
        f"Distribusi jumlah objek/ko-kemunculan memakai {len(count_images_df):,} gambar berlabel lengkap; "
        f"brightness berhasil diukur pada {len(brightness_df):,} gambar. "
        f"Audit menemukan {int(critical_issues['occurrences'].sum()):,} kejadian kritis. "
        "Angka tersebut mendeskripsikan data dan cakupan pemeriksaan; tidak membuktikan "
        "metode KD tertentu lebih unggul. **Laporan angka dan batas interpretasi:** "
        f"`{OUTPUT_DIR / 'reports/eda_summary.md'}`."
    )
)
